In [0]:
"""
fact_trades
fact_p2p_transactions
fact_balance_snapshots
fact_asset_movements
fact_daily_prices
"""

'\nfact_trades\nfact_p2p_transactions\nfact_balance_snapshots\nfact_asset_movements\nfact_daily_prices\n'

####Fact trades

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

catalog = "crypto_lakehouse"
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"

binance_trades_df = spark.table(
    f"{catalog}.{silver_schema}.binance_trades"
)

giottus_trades_df = spark.table(
    f"{catalog}.{silver_schema}.giottus_trades"
)

coindcx_trades_df = spark.table(
    f"{catalog}.{silver_schema}.coindcx_trades"
)

mudrex_trades_df = spark.table(
    f"{catalog}.{silver_schema}.mudrex_trades"
)

print("Binance trade rows:", binance_trades_df.count())
print("Giottus trade rows:", giottus_trades_df.count())
print("CoinDCX trade rows:", coindcx_trades_df.count())
print("Mudrex trade rows:", mudrex_trades_df.count())

binance_trades_df.printSchema()
giottus_trades_df.printSchema()
coindcx_trades_df.printSchema()
mudrex_trades_df.printSchema()

Binance trade rows: 16
Giottus trade rows: 2
CoinDCX trade rows: 1
Mudrex trade rows: 37
root
 |-- transaction_id: string (nullable = true)
 |-- trade_source: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- side: string (nullable = true)
 |-- asset_spent: string (nullable = true)
 |-- amount_spent: decimal(38,18) (nullable = true)
 |-- asset_received: string (nullable = true)
 |-- amount_received: decimal(38,18) (nullable = true)
 |-- fee_amount: decimal(38,18) (nullable = true)
 |-- fee_asset: string (nullable = true)
 |-- trade_time: timestamp (nullable = true)
 |-- exchange: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- trade_key: string (nullable = true)
 |-- trade_date: date (nullable = true)
 |-- silver_processed_at: timestamp (nullable = true)

root
 |-- transaction_id: string (nullable = true)
 |-- trade_source: string (nullable = true)
 |-- symbol: string (nullable = true)
 |--

In [0]:
mudrex_trades_df = (
    mudrex_trades_df
    .withColumn(
        "fee_amount",
        col("fee_amount").cast("decimal(38,18)")
    )
)

mudrex_trades_df.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- trade_source: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- side: string (nullable = true)
 |-- asset_spent: string (nullable = true)
 |-- amount_spent: decimal(38,18) (nullable = true)
 |-- asset_received: string (nullable = true)
 |-- amount_received: decimal(38,18) (nullable = true)
 |-- fee_amount: decimal(38,18) (nullable = true)
 |-- fee_asset: string (nullable = true)
 |-- trade_time: timestamp (nullable = true)
 |-- exchange: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- trade_key: string (nullable = true)
 |-- trade_date: date (nullable = true)
 |-- silver_processed_at: timestamp (nullable = true)



In [0]:
silver_trades_df = (
    binance_trades_df
    .unionByName(giottus_trades_df)
    .unionByName(coindcx_trades_df)
    .unionByName(mudrex_trades_df)
    .dropDuplicates(["trade_key"])
)

print(
    "Combined Silver trade rows:",
    silver_trades_df.count()
)

display(
    silver_trades_df
    .select(
        "exchange",
        "trade_source",
        "transaction_id",
        "symbol",
        "asset_spent",
        "amount_spent",
        "asset_received",
        "amount_received",
        "fee_amount",
        "fee_asset",
        "trade_time"
    )
    .orderBy(
        col("trade_time").desc()
    )
)

Combined Silver trade rows: 56


exchange,trade_source,transaction_id,symbol,asset_spent,amount_spent,asset_received,amount_received,fee_amount,fee_asset,trade_time
Binance,Spot,6636792208,BTCUSDT,USDT,3.164860000000000000,BTC,0.000040000000000000,4.0000000000E-8,BTC,2026-08-30T18:32:38.003Z
Binance,Spot,6636792207,BTCUSDT,USDT,5.538505000000000000,BTC,0.000070000000000000,7.0000000000E-8,BTC,2026-08-30T18:32:38.003Z
Binance,Spot,4255932876,ETHUSDT,USDT,7.513640000000000000,ETH,0.004000000000000000,0.000004000000000000,ETH,2026-08-05T15:26:48.425Z
Binance,Spot,6497931425,BTCUSDT,USDT,6.426600000000000000,BTC,0.000100000000000000,1.00000000000E-7,BTC,2026-07-11T19:40:10.981Z
Binance,Spot,60681896,ONDOUSDT,USDT,6.168930000000000000,ONDO,18.300000000000000000,0.018300000000000000,ONDO,2026-07-04T22:24:18.443Z
Binance,Spot,472919659,LINKUSDT,USDT,5.068160000000000000,LINK,0.640000000000000000,0.000640000000000000,LINK,2026-06-14T09:38:33.928Z
Binance,Spot,472919660,LINKUSDT,USDT,0.950280000000000000,LINK,0.120000000000000000,0.000120000000000000,LINK,2026-06-14T09:38:33.928Z
Binance,Spot,523831127,SUIUSDT,USDT,6.071760000000000000,SUI,8.100000000000000000,0.008100000000000000,SUI,2026-06-11T15:50:18.493Z
Binance,Spot,4121758522,ETHUSDT,USDT,4.974870000000000000,ETH,0.003000000000000000,0.000003000000000000,ETH,2026-06-11T06:28:07.152Z
Giottus,Spot,228465533,SOL/INR,INR,100.738020000000000000,SOL,0.015058000000000000,0.000060000000000000,SOL,2026-06-09T21:54:49.000Z


In [0]:
fact_trades_df = (
    silver_trades_df
    .select(
        col("trade_key"),

        date_format(
            col("trade_time"),
            "yyyyMMdd"
        )
        .cast("integer")
        .alias("date_key"),

        upper(
            col("exchange")
        ).alias("exchange_key"),

        upper(
            col("asset_spent")
        ).alias("spent_asset_key"),

        upper(
            col("asset_received")
        ).alias("received_asset_key"),

        col("transaction_id"),
        col("trade_source"),
        col("symbol"),
        col("side"),

        col("amount_spent"),
        col("amount_received"),

        col("fee_amount"),
        col("fee_asset"),

        col("trade_time"),
        col("_source_file"),
        col("bronze_ingested_at")
    )
    .withColumn(
        "gold_processed_at",
        current_timestamp()
    )
)

print(
    "Gold trade rows:",
    fact_trades_df.count()
)

display(
    fact_trades_df.orderBy(
        col("trade_time").desc()
    )
)

Gold trade rows: 56


trade_key,date_key,exchange_key,spent_asset_key,received_asset_key,transaction_id,trade_source,symbol,side,amount_spent,amount_received,fee_amount,fee_asset,trade_time,_source_file,bronze_ingested_at,gold_processed_at
Binance|Spot|BTCUSDT|6636792208,20260830,BINANCE,USDT,BTC,6636792208,Spot,BTCUSDT,BUY,3.164860000000000000,0.000040000000000000,4.0000000000E-8,BTC,2026-08-30T18:32:38.003Z,s3://l1l2bucket/data/raw/binance/trades/BTCUSDT/trades_20260925T182105_429433Z_77f23558.json,2026-10-02T14:21:33.930Z,2026-10-02T16:32:20.439Z
Binance|Spot|BTCUSDT|6636792207,20260830,BINANCE,USDT,BTC,6636792207,Spot,BTCUSDT,BUY,5.538505000000000000,0.000070000000000000,7.0000000000E-8,BTC,2026-08-30T18:32:38.003Z,s3://l1l2bucket/data/raw/binance/trades/BTCUSDT/trades_20260925T182105_429433Z_77f23558.json,2026-10-02T14:21:33.930Z,2026-10-02T16:32:20.439Z
Binance|Spot|ETHUSDT|4255932876,20260805,BINANCE,USDT,ETH,4255932876,Spot,ETHUSDT,BUY,7.513640000000000000,0.004000000000000000,0.000004000000000000,ETH,2026-08-05T15:26:48.425Z,s3://l1l2bucket/data/raw/binance/trades/ETHUSDT/trades_20260925T182105_922892Z_d577bdb1.json,2026-10-02T14:21:33.930Z,2026-10-02T16:32:20.439Z
Binance|Spot|BTCUSDT|6497931425,20260711,BINANCE,USDT,BTC,6497931425,Spot,BTCUSDT,BUY,6.426600000000000000,0.000100000000000000,1.00000000000E-7,BTC,2026-07-11T19:40:10.981Z,s3://l1l2bucket/data/raw/binance/trades/BTCUSDT/trades_20260925T182105_429433Z_77f23558.json,2026-10-02T14:21:33.930Z,2026-10-02T16:32:20.439Z
Binance|Spot|ONDOUSDT|60681896,20260704,BINANCE,USDT,ONDO,60681896,Spot,ONDOUSDT,BUY,6.168930000000000000,18.300000000000000000,0.018300000000000000,ONDO,2026-07-04T22:24:18.443Z,s3://l1l2bucket/data/raw/binance/trades/ONDOUSDT/trades_20260925T182107_841454Z_e13b5606.json,2026-10-02T14:21:33.930Z,2026-10-02T16:32:20.439Z
Binance|Spot|LINKUSDT|472919659,20260614,BINANCE,USDT,LINK,472919659,Spot,LINKUSDT,BUY,5.068160000000000000,0.640000000000000000,0.000640000000000000,LINK,2026-06-14T09:38:33.928Z,s3://l1l2bucket/data/raw/binance/trades/LINKUSDT/trades_20260925T182107_371435Z_084ca1ab.json,2026-10-02T14:21:33.930Z,2026-10-02T16:32:20.439Z
Binance|Spot|LINKUSDT|472919660,20260614,BINANCE,USDT,LINK,472919660,Spot,LINKUSDT,BUY,0.950280000000000000,0.120000000000000000,0.000120000000000000,LINK,2026-06-14T09:38:33.928Z,s3://l1l2bucket/data/raw/binance/trades/LINKUSDT/trades_20260925T182107_371435Z_084ca1ab.json,2026-10-02T14:21:33.930Z,2026-10-02T16:32:20.439Z
Binance|Spot|SUIUSDT|523831127,20260611,BINANCE,USDT,SUI,523831127,Spot,SUIUSDT,BUY,6.071760000000000000,8.100000000000000000,0.008100000000000000,SUI,2026-06-11T15:50:18.493Z,s3://l1l2bucket/data/raw/binance/trades/SUIUSDT/trades_20260925T182106_890100Z_aa258a00.json,2026-10-02T14:21:33.930Z,2026-10-02T16:32:20.439Z
Binance|Spot|ETHUSDT|4121758522,20260611,BINANCE,USDT,ETH,4121758522,Spot,ETHUSDT,BUY,4.974870000000000000,0.003000000000000000,0.000003000000000000,ETH,2026-06-11T06:28:07.152Z,s3://l1l2bucket/data/raw/binance/trades/ETHUSDT/trades_20260925T182105_922892Z_d577bdb1.json,2026-10-02T14:21:33.930Z,2026-10-02T16:32:20.439Z
Giottus|Spot|SOL/INR|228465533,20260609,GIOTTUS,INR,SOL,228465533,Spot,SOL/INR,BUY,100.738020000000000000,0.015058000000000000,0.000060000000000000,SOL,2026-06-09T21:54:49.000Z,s3://l1l2bucket/data/raw/giottus/trades/SOLINR/trades_20260925T182206_079516Z_4a735758.json,2026-09-25T18:26:45.897Z,2026-10-02T16:32:20.439Z


In [0]:
(
    fact_trades_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{gold_schema}.fact_trades"
    )
)

display(
    spark.table(
        f"{catalog}.{gold_schema}.fact_trades"
    )
)

trade_key,date_key,exchange_key,spent_asset_key,received_asset_key,transaction_id,trade_source,symbol,side,amount_spent,amount_received,fee_amount,fee_asset,trade_time,_source_file,bronze_ingested_at,gold_processed_at
Binance|Spot|BTCUSDT|6636792207,20260830,BINANCE,USDT,BTC,6636792207,Spot,BTCUSDT,BUY,5.538505000000000000,0.000070000000000000,7.0000000000E-8,BTC,2026-08-30T18:32:38.003Z,s3://l1l2bucket/data/raw/binance/trades/BTCUSDT/trades_20260925T182105_429433Z_77f23558.json,2026-10-02T14:21:33.930Z,2026-10-02T16:32:23.006Z
Binance|Spot|SOLUSDT|1972197390,20260605,BINANCE,USDT,SOL,1972197390,Spot,SOLUSDT,BUY,5.128680000000000000,0.079000000000000000,0.000079000000000000,SOL,2026-06-05T14:14:53.697Z,s3://l1l2bucket/data/raw/binance/trades/SOLUSDT/trades_20260925T182106_413092Z_d4c5c822.json,2026-10-02T14:21:33.930Z,2026-10-02T16:32:23.006Z
Binance|Spot|BTCUSDT|6636792208,20260830,BINANCE,USDT,BTC,6636792208,Spot,BTCUSDT,BUY,3.164860000000000000,0.000040000000000000,4.0000000000E-8,BTC,2026-08-30T18:32:38.003Z,s3://l1l2bucket/data/raw/binance/trades/BTCUSDT/trades_20260925T182105_429433Z_77f23558.json,2026-10-02T14:21:33.930Z,2026-10-02T16:32:23.006Z
Binance|Spot|ETHUSDT|4255932876,20260805,BINANCE,USDT,ETH,4255932876,Spot,ETHUSDT,BUY,7.513640000000000000,0.004000000000000000,0.000004000000000000,ETH,2026-08-05T15:26:48.425Z,s3://l1l2bucket/data/raw/binance/trades/ETHUSDT/trades_20260925T182105_922892Z_d577bdb1.json,2026-10-02T14:21:33.930Z,2026-10-02T16:32:23.006Z
Binance|Spot|LINKUSDT|472919660,20260614,BINANCE,USDT,LINK,472919660,Spot,LINKUSDT,BUY,0.950280000000000000,0.120000000000000000,0.000120000000000000,LINK,2026-06-14T09:38:33.928Z,s3://l1l2bucket/data/raw/binance/trades/LINKUSDT/trades_20260925T182107_371435Z_084ca1ab.json,2026-10-02T14:21:33.930Z,2026-10-02T16:32:23.006Z
Binance|Spot|ETHUSDT|4121758522,20260611,BINANCE,USDT,ETH,4121758522,Spot,ETHUSDT,BUY,4.974870000000000000,0.003000000000000000,0.000003000000000000,ETH,2026-06-11T06:28:07.152Z,s3://l1l2bucket/data/raw/binance/trades/ETHUSDT/trades_20260925T182105_922892Z_d577bdb1.json,2026-10-02T14:21:33.930Z,2026-10-02T16:32:23.006Z
Binance|Spot|LINKUSDT|472919659,20260614,BINANCE,USDT,LINK,472919659,Spot,LINKUSDT,BUY,5.068160000000000000,0.640000000000000000,0.000640000000000000,LINK,2026-06-14T09:38:33.928Z,s3://l1l2bucket/data/raw/binance/trades/LINKUSDT/trades_20260925T182107_371435Z_084ca1ab.json,2026-10-02T14:21:33.930Z,2026-10-02T16:32:23.006Z
Binance|Spot|SUIUSDT|523831127,20260611,BINANCE,USDT,SUI,523831127,Spot,SUIUSDT,BUY,6.071760000000000000,8.100000000000000000,0.008100000000000000,SUI,2026-06-11T15:50:18.493Z,s3://l1l2bucket/data/raw/binance/trades/SUIUSDT/trades_20260925T182106_890100Z_aa258a00.json,2026-10-02T14:21:33.930Z,2026-10-02T16:32:23.006Z
Binance|Spot|SOLUSDT|1972197389,20260605,BINANCE,USDT,SOL,1972197389,Spot,SOLUSDT,BUY,1.752840000000000000,0.027000000000000000,0.000027000000000000,SOL,2026-06-05T14:14:53.697Z,s3://l1l2bucket/data/raw/binance/trades/SOLUSDT/trades_20260925T182106_413092Z_d4c5c822.json,2026-10-02T14:21:33.930Z,2026-10-02T16:32:23.006Z
Binance|Spot|SOLUSDT|1972197391,20260605,BINANCE,USDT,SOL,1972197391,Spot,SOLUSDT,BUY,0.584280000000000000,0.009000000000000000,0.000009000000000000,SOL,2026-06-05T14:14:53.697Z,s3://l1l2bucket/data/raw/binance/trades/SOLUSDT/trades_20260925T182106_413092Z_d4c5c822.json,2026-10-02T14:21:33.930Z,2026-10-02T16:32:23.006Z


####fact p2p transactions

In [0]:
silver_p2p_df = spark.table(
    f"{catalog}.{silver_schema}.binance_p2p_transactions"
)

fact_p2p_df = (
    silver_p2p_df
    .select(
        concat_ws(
            "|",
            upper(col("exchange")),
            lit("P2P"),
            col("p2p_order_id")
        ).alias("transaction_key"),

        date_format(
            col("transaction_time"),
            "yyyyMMdd"
        )
        .cast("integer")
        .alias("date_key"),

        upper(col("exchange"))
        .alias("exchange_key"),

        upper(
            col("asset")
        ).alias("asset_key"),

        col("p2p_order_id"),
        col("trade_type"),

        upper(
            col("fiat")
        ).alias('fiat_currency'),

        col("asset_amount"),
        col("fiat_amount"),
        col("unit_price"),
        col("fee_amount"),
        col("net_asset_amount"),

        col("status"),
        col("is_completed"),
        col("transaction_time"),
        col("payment_method"),
        col("_source_file"),
        col("bronze_ingested_at")
    )
    .withColumn(
        "gold_processed_at",
        current_timestamp()
    )
)

print(
    "Gold P2p rows:",
    fact_p2p_df.count()
)

display(
    fact_p2p_df.orderBy(
        col("transaction_time").desc()
    )
)

Gold P2p rows: 15


transaction_key,date_key,exchange_key,asset_key,p2p_order_id,trade_type,fiat_currency,asset_amount,fiat_amount,unit_price,fee_amount,net_asset_amount,status,is_completed,transaction_time,payment_method,_source_file,bronze_ingested_at,gold_processed_at
BINANCE|P2P|22927112581471313920,20260829,BINANCE,USDT,22927112581471313920,BUY,INR,5.330000000000000000,550.000000000000000000,103.000000000000000000,0.060000000000000000,5.270000000000000000,COMPLETED,true,2026-08-29T20:21:08.551Z,UPI,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,2026-10-02T16:32:27.883Z
BINANCE|P2P|22924149347226902528,20260821,BINANCE,USDT,22924149347226902528,BUY,INR,3.860000000000000000,400.000000000000000000,103.500000000000000000,0.060000000000000000,3.800000000000000000,COMPLETED,true,2026-08-21T16:06:18.475Z,UPI,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,2026-10-02T16:32:27.883Z
BINANCE|P2P|22918338709083369472,20260805,BINANCE,USDT,22918338709083369472,BUY,INR,3.880000000000000000,400.000000000000000000,102.940000000000000000,0.060000000000000000,3.820000000000000000,COMPLETED,true,2026-08-05T15:16:54.361Z,UPI,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,2026-10-02T16:32:27.883Z
BINANCE|P2P|22910348341135761408,20260714,BINANCE,USDT,22910348341135761408,BUY,INR,3.420000000000000000,350.000000000000000000,102.300000000000000000,0.060000000000000000,3.360000000000000000,COMPLETED,true,2026-07-14T14:06:02.235Z,UPIQRCode,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,2026-10-02T16:32:27.883Z
BINANCE|P2P|22909209175754317824,20260711,BINANCE,USDT,22909209175754317824,BUY,INR,3.420000000000000000,350.000000000000000000,102.200000000000000000,0.060000000000000000,3.360000000000000000,COMPLETED,true,2026-07-11T10:39:24.047Z,UPIQRCode,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,2026-10-02T16:32:27.883Z
BINANCE|P2P|22907162745527037952,20260705,BINANCE,USDT,22907162745527037952,BUY,INR,3.500000000000000000,359.000000000000000000,102.300000000000000000,0.060000000000000000,3.440000000000000000,COMPLETED,true,2026-07-05T19:07:37.023Z,UPIQRCode,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,2026-10-02T16:32:27.883Z
BINANCE|P2P|22906848752644837376,20260704,BINANCE,USDT,22906848752644837376,BUY,INR,2.910000000000000000,300.000000000000000000,102.990000000000000000,0.060000000000000000,2.850000000000000000,COMPLETED,true,2026-07-04T22:19:55.318Z,UPI,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,2026-10-02T16:32:27.883Z
BINANCE|P2P|22906848402065932288,20260704,BINANCE,USDT,22906848402065932288,BUY,INR,3.390000000000000000,350.000000000000000000,103.000000000000000000,0.060000000000000000,3.330000000000000000,COMPLETED,true,2026-07-04T22:18:31.749Z,UPIQRCode,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,2026-10-02T16:32:27.883Z
BINANCE|P2P|22899404533812277248,20260614,BINANCE,USDT,22899404533812277248,BUY,INR,6.080000000000000000,630.000000000000000000,103.540000000000000000,0.060000000000000000,6.020000000000000000,COMPLETED,true,2026-06-14T09:19:15.226Z,UPI,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,2026-10-02T16:32:27.883Z
BINANCE|P2P|22898413188949139456,20260611,BINANCE,USDT,22898413188949139456,BUY,INR,6.050000000000000000,630.000000000000000000,103.990000000000000000,0.060000000000000000,5.990000000000000000,COMPLETED,true,2026-06-11T15:40:00.189Z,UPI,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,2026-10-02T14:22:07.631Z,2026-10-0

In [0]:
(
    fact_p2p_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{gold_schema}.fact_p2p_transactions"
    )
)

saved_fact_p2p_df = spark.table(
    f"{catalog}.{gold_schema}.fact_p2p_transactions"
)

print(
    "Saved Gold P2P rows:",
    saved_fact_p2p_df.count()
)

Saved Gold P2P rows: 15


####Fact balance snapshots

In [0]:
# Load balance snapshots

spot_snapshot_balances_df = spark.table(
    f"{catalog}.{silver_schema}.binance_account_balances"
)

funding_snapshot_balances_df = spark.table(
    f"{catalog}.{silver_schema}.binance_funding_account_balances"
)

giottus_snapshot_balances_df = spark.table(
    f"{catalog}.{silver_schema}.giottus_account_balances"
)

coindcx_snapshot_balances_df = spark.table(
    f"{catalog}.{silver_schema}.coindcx_account_balances"
)


# Prepare Binance Spot balances

spot_fact_source_df = (
    spot_snapshot_balances_df
    .select(
        col("snapshot_id"),
        col("account_id"),
        lit("SPOT").alias("wallet_type"),
        lit("BINANCE_SPOT").alias("wallet_key"),
        upper(col("asset")).alias("asset_key"),

        col("free_balance")
        .cast("decimal(38,18)")
        .alias("free_balance"),

        col("locked_balance")
        .cast("decimal(38,18)")
        .alias("locked_balance"),

        lit(0)
        .cast("decimal(38,18)")
        .alias("frozen_balance"),

        lit(0)
        .cast("decimal(38,18)")
        .alias("withdrawing_balance"),

        col("total_balance")
        .cast("decimal(38,18)")
        .alias("total_balance"),

        upper(col("exchange"))
        .alias("exchange_key"),

        coalesce(
            to_timestamp(
                regexp_extract(
                    col("snapshot_id"),
                    r"account_(\d{8}T\d{6}_\d{6}Z)",
                    1
                ),
                "yyyyMMdd'T'HHmmss_SSSSSS'Z'"
            ),
            col("bronze_ingested_at")
        ).alias("snapshot_time"),

        col("_source_file")
    )
)


# Prepare Binance Funding balances

funding_fact_source_df = (
    funding_snapshot_balances_df
    .select(
        col("snapshot_id"),

        lit(None)
        .cast("string")
        .alias("account_id"),

        upper(col("wallet_type"))
        .alias("wallet_type"),

        lit("BINANCE_FUNDING")
        .alias("wallet_key"),

        upper(col("asset"))
        .alias("asset_key"),

        col("free_balance")
        .cast("decimal(38,18)")
        .alias("free_balance"),

        col("locked_balance")
        .cast("decimal(38,18)")
        .alias("locked_balance"),

        col("frozen_balance")
        .cast("decimal(38,18)")
        .alias("frozen_balance"),

        col("withdrawing_balance")
        .cast("decimal(38,18)")
        .alias("withdrawing_balance"),

        col("total_balance")
        .cast("decimal(38,18)")
        .alias("total_balance"),

        upper(col("exchange"))
        .alias("exchange_key"),

        coalesce(
            col("snapshot_time"),
            col("bronze_ingested_at")
        ).alias("snapshot_time"),

        col("_source_file")
    )
)


# Prepare Giottus Spot balances

giottus_fact_source_df = (
    giottus_snapshot_balances_df
    .select(
        col("snapshot_id"),
        col("account_id"),
        lit("SPOT").alias("wallet_type"),
        lit("GIOTTUS_SPOT").alias("wallet_key"),
        upper(col("asset")).alias("asset_key"),

        col("free_balance")
        .cast("decimal(38,18)")
        .alias("free_balance"),

        col("locked_balance")
        .cast("decimal(38,18)")
        .alias("locked_balance"),

        lit(0)
        .cast("decimal(38,18)")
        .alias("frozen_balance"),

        lit(0)
        .cast("decimal(38,18)")
        .alias("withdrawing_balance"),

        col("total_balance")
        .cast("decimal(38,18)")
        .alias("total_balance"),

        upper(col("exchange"))
        .alias("exchange_key"),

        col("bronze_ingested_at")
        .alias("snapshot_time"),

        col("_source_file")
    )
)


# Prepare CoinDCX Spot balances

coindcx_fact_source_df = (
    coindcx_snapshot_balances_df
    .select(
        col("snapshot_id"),
        col("account_id"),
        lit("SPOT").alias("wallet_type"),
        lit("COINDCX_SPOT").alias("wallet_key"),
        upper(col("asset")).alias("asset_key"),

        col("free_balance")
        .cast("decimal(38,18)")
        .alias("free_balance"),

        col("locked_balance")
        .cast("decimal(38,18)")
        .alias("locked_balance"),

        lit(0)
        .cast("decimal(38,18)")
        .alias("frozen_balance"),

        lit(0)
        .cast("decimal(38,18)")
        .alias("withdrawing_balance"),

        col("total_balance")
        .cast("decimal(38,18)")
        .alias("total_balance"),

        upper(col("exchange"))
        .alias("exchange_key"),

        col("bronze_ingested_at")
        .alias("snapshot_time"),

        col("_source_file")
    )
)


# Combine all wallet snapshots

combined_snapshot_balances_df = (
    spot_fact_source_df
    .unionByName(
        funding_fact_source_df
    )
    .unionByName(
        giottus_fact_source_df
    )
    .unionByName(
        coindcx_fact_source_df
    )
)


# Keep only assets listed in dim_asset

tracked_assets_df = (
    spark.table(
        f"{catalog}.{gold_schema}.dim_asset"
    )
    .select("asset_key")
    .dropDuplicates()
)


# Create the combined balance-snapshot fact dataframe

fact_balance_snapshots_df = (
    combined_snapshot_balances_df
    .join(
        tracked_assets_df,
        ["asset_key"],
        "inner"
    )
    .withColumn(
        "date_key",
        date_format(
            col("snapshot_time"),
            "yyyyMMdd"
        ).cast("integer")
    )
    .withColumn(
        "balance_key",
        concat_ws(
            "|",
            col("wallet_key"),
            col("snapshot_id"),
            coalesce(
                col("account_id"),
                lit("NO_ACCOUNT")
            ),
            col("asset_key")
        )
    )
    .dropDuplicates([
        "balance_key"
    ])
    .withColumn(
        "gold_processed_at",
        current_timestamp()
    )
    .select(
        "balance_key",
        "date_key",
        "exchange_key",
        "wallet_key",
        "wallet_type",
        "asset_key",
        "snapshot_id",
        "account_id",
        "free_balance",
        "locked_balance",
        "frozen_balance",
        "withdrawing_balance",
        "total_balance",
        "snapshot_time",
        "_source_file",
        "gold_processed_at"
    )
)

print(
    "Combined balance snapshot rows:",
    fact_balance_snapshots_df.count()
)

display(
    fact_balance_snapshots_df
    .groupBy(
        "exchange_key",
        "wallet_key"
    )
    .count()
    .orderBy(
        "exchange_key",
        "wallet_key"
    )
)

display(
    fact_balance_snapshots_df
    .filter(
        col("exchange_key") == "COINDCX"
    )
)

Combined balance snapshot rows: 19


exchange_key,wallet_key,count
BINANCE,BINANCE_SPOT,14
COINDCX,COINDCX_SPOT,4
GIOTTUS,GIOTTUS_SPOT,1


balance_key,date_key,exchange_key,wallet_key,wallet_type,asset_key,snapshot_id,account_id,free_balance,locked_balance,frozen_balance,withdrawing_balance,total_balance,snapshot_time,_source_file,gold_processed_at
COINDCX_SPOT|balances_20260925T181928_277012Z_c83f7988.json|NO_ACCOUNT|SOL,20260925,COINDCX,COINDCX_SPOT,SPOT,SOL,balances_20260925T181928_277012Z_c83f7988.json,null,0.121948825900600000,0E-18,0E-18,0E-18,0.121948825900600000,2026-09-25T18:26:39.618Z,s3://l1l2bucket/data/raw/coindcx/balances/balances_20260925T181928_277012Z_c83f7988.json,2026-10-02T16:32:37.000Z
COINDCX_SPOT|balances_20260925T181928_277012Z_c83f7988.json|NO_ACCOUNT|USDT,20260925,COINDCX,COINDCX_SPOT,SPOT,USDT,balances_20260925T181928_277012Z_c83f7988.json,null,0E-18,0E-18,0E-18,0E-18,0E-18,2026-09-25T18:26:39.618Z,s3://l1l2bucket/data/raw/coindcx/balances/balances_20260925T181928_277012Z_c83f7988.json,2026-10-02T16:32:37.000Z
COINDCX_SPOT|balances_20260925T181928_277012Z_c83f7988.json|NO_ACCOUNT|INR,20260925,COINDCX,COINDCX_SPOT,SPOT,INR,balances_20260925T181928_277012Z_c83f7988.json,null,0.034810000000000000,0E-18,0E-18,0E-18,0.034810000000000000,2026-09-25T18:26:39.618Z,s3://l1l2bucket/data/raw/coindcx/balances/balances_20260925T181928_277012Z_c83f7988.json,2026-10-02T16:32:37.000Z
COINDCX_SPOT|balances_20260925T181928_277012Z_c83f7988.json|NO_ACCOUNT|BTC,20260925,COINDCX,COINDCX_SPOT,SPOT,BTC,balances_20260925T181928_277012Z_c83f7988.json,null,0E-18,0E-18,0E-18,0E-18,0E-18,2026-09-25T18:26:39.618Z,s3://l1l2bucket/data/raw/coindcx/balances/balances_20260925T181928_277012Z_c83f7988.json,2026-10-02T16:32:37.000Z


In [0]:
(
    fact_balance_snapshots_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{gold_schema}.fact_balance_snapshots"
    )
)

####Fact asset movements

In [0]:
mudrex_deposit_movements_df = (
    mudrex_deposits_df
    .filter(
        upper(col("status")) == "COMPLETED"
    )
    .select(
        concat_ws(
            "|",
            upper(col("exchange")),
            lit("DEPOSIT"),
            col("deposit_id")
        ).alias("movement_key"),

        date_format(
            col("deposit_time"),
            "yyyyMMdd"
        ).cast("integer").alias("date_key"),

        upper(col("exchange")).alias(
            "exchange_key"
        ),

        upper(col("asset")).alias(
            "asset_key"
        ),

        lit("DEPOSIT").alias(
            "movement_type"
        ),

        col("deposit_time").alias(
            "movement_time"
        ),

        (
            col("amount")
            - coalesce(
                col("fee_amount"),
                lit(0).cast("decimal(38,18)")
            )
        ).alias("quantity_change")
    )
    .dropDuplicates([
        "movement_key"
    ])
)

print(
    "Mudrex deposit movement rows:",
    mudrex_deposit_movements_df.count()
)

display(
    mudrex_deposit_movements_df
    .orderBy("movement_time")
)

Mudrex deposit movement rows: 14


movement_key,date_key,exchange_key,asset_key,movement_type,movement_time,quantity_change
MUDREX|DEPOSIT|955bce71-177c-43d1-ba3f-2d6e72ff046c,20250530,MUDREX,USDT,DEPOSIT,2025-05-30T11:34:12.000Z,5.69022419000000000
MUDREX|DEPOSIT|019824a9-e370-770d-b122-bc0cd3e3c68f,20250719,MUDREX,USDT,DEPOSIT,2025-07-19T21:49:30.000Z,5.62499500000000000
MUDREX|DEPOSIT|01982e32-2a01-76bf-b213-928dda2fcf41,20250721,MUDREX,USDT,DEPOSIT,2025-07-21T18:14:56.000Z,11.24999000000000000
MUDREX|DEPOSIT|01993a31-94dd-7a52-81e4-a4dc78d6cb76,20250911,MUDREX,USDT,DEPOSIT,2025-09-11T19:12:32.000Z,5.67174570000000000
MUDREX|DEPOSIT|0199d2b2-eee0-7807-9910-57c54b22a02e,20251011,MUDREX,USDT,DEPOSIT,2025-10-11T09:56:06.000Z,5.20851774530233700
MUDREX|DEPOSIT|019a51e4-d65e-7aad-9c0b-183cbf285eb1,20251105,MUDREX,USDT,DEPOSIT,2025-11-05T02:42:23.000Z,5.45328961748542900
MUDREX|DEPOSIT|019ab2ba-2ec6-7c74-b3a7-38c69983fbec,20251123,MUDREX,USDT,DEPOSIT,2025-11-23T21:58:57.000Z,10.60815368196303700
MUDREX|DEPOSIT|019c1577-cd7c-7010-8b5c-aecb6b1d5982,20260131,MUDREX,USDT,DEPOSIT,2026-01-31T19:11:39.000Z,10.45647558386300000
MUDREX|DEPOSIT|019c469b-8982-730d-a38f-3613fed7dfe5,20260210,MUDREX,INR,DEPOSIT,2026-02-10T08:12:04.000Z,1200.00000000000000000
MUDREX|DEPOSIT|019c46a2-7f08-79c7-8e1c-679f9ed7d73b,20260210,MUDREX,USDT,DEPOSIT,2026-02-10T08:19:40.000Z,12.31249999999920000


In [0]:
mudrex_withdrawal_movements_df = (
    mudrex_withdrawals_df
    .filter(
        upper(col("status")) == "COMPLETED"
    )
    .select(
        concat_ws(
            "|",
            upper(col("exchange")),
            lit("WITHDRAWAL"),
            col("withdrawal_id")
        ).alias("movement_key"),

        date_format(
            col("withdrawal_time"),
            "yyyyMMdd"
        ).cast("integer").alias("date_key"),

        upper(col("exchange")).alias(
            "exchange_key"
        ),

        upper(col("asset")).alias(
            "asset_key"
        ),

        lit("WITHDRAWAL").alias(
            "movement_type"
        ),

        col("withdrawal_time").alias(
            "movement_time"
        ),

        (
            -(
                col("amount")
                + coalesce(
                    col("transaction_fee"),
                    lit(0).cast("decimal(38,18)")
                )
                + coalesce(
                    col("tds_inr"),
                    lit(0).cast("decimal(38,18)")
                )
            )
        ).alias("quantity_change")
    )
    .dropDuplicates([
        "movement_key"
    ])
)

print(
    "Mudrex withdrawal movement rows:",
    mudrex_withdrawal_movements_df.count()
)

display(mudrex_withdrawal_movements_df)

Mudrex withdrawal movement rows: 1


movement_key,date_key,exchange_key,asset_key,movement_type,movement_time,quantity_change
MUDREX|WITHDRAWAL|019c46a1-d206-78c8-8404-4e7b19c06643,20260210,MUDREX,INR,WITHDRAWAL,2026-02-10T08:18:56.000Z,-1200.0000000000000000


In [0]:
display(
    mudrex_earn_df
    .filter(
        (upper(col("transaction_type")) == "INTEREST")
        & (upper(col("status")) == "SUCCESSFUL")
    )
    .select(
        "earn_id",
        "asset",
        "quantity",
        "earn_wallet_change",
        "is_reward",
        "earn_time"
    )
    .orderBy(
        col("earn_time").desc()
    )
    .limit(10)
)

earn_id,asset,quantity,earn_wallet_change,is_reward,earn_time
2a79ef66af6313e0e9f1d31bd328959b818ac3e7197789b146428a6d83231319,SOL,0.000519650000000000,0.000519650000000000,true,2026-03-29T12:00:04.000Z
abb50795b8193e3fd6d22504870f25bdf7392e175a8b54d463ecc88ac750e914,ETH,0.000011970000000000,0.000011970000000000,true,2026-03-29T11:59:31.000Z
ba468f7d2db1b2c2584e5fc1592a5352d9055c6d3d7724c5da8a2081ae0929e1,ETH,0.000001480000000000,0.000001480000000000,true,2026-03-15T17:14:03.000Z
f4c084ee3a1b8bb62b18526045ab91e54930c40ab0048f5007cb4109c31fa257,BTC,1.0000000000E-8,1.0000000000E-8,true,2026-03-13T14:01:43.000Z
4ed66fc2f5b2a2036602c4e04ee5801d5d3876e06d585b7b1033b27ab464b3a5,SOL,0.000250150000000000,0.000250150000000000,true,2026-03-13T14:01:24.000Z
275659bcee012015098ceb98df3f26b8c7f97d61a37a39e197f1bca7be377e61,ETH,0.000015180000000000,0.000015180000000000,true,2026-03-13T14:00:46.000Z
d89c3323219f6c1dead45ff4bf5a53b1b06929d2f7e4807c29d1e6cd0840e45d,SUI,0.000312230000000000,0.000312230000000000,true,2026-03-09T22:05:58.000Z
22d0f67b5350893753b6dd2dc8a79d8edf1a9f61c0001224e201ed163072d895,SOL,0.000109750000000000,0.000109750000000000,true,2026-03-02T21:08:00.000Z
c42cc6d20c1526937bc93fa221d3ab21d1cb5f9b3066e8a962c4c29d66f8229e,SOL,0.000093030000000000,0.000093030000000000,true,2026-02-25T20:38:08.000Z
599accdb1426ff9e095dbcd004c3da061fab421dd6dbf4f93b542d148859c102,SUI,0.000023380000000000,0.000023380000000000,true,2026-02-21T21:50:29.000Z


In [0]:
mudrex_earn_reward_movements_df = (
    mudrex_earn_df
    .filter(
        (upper(col("transaction_type")) == "INTEREST")
        & (upper(col("status")) == "SUCCESSFUL")
        & (col("is_reward") == True)
    )
    .select(
        concat_ws(
            "|",
            upper(col("exchange")),
            lit("EARN_REWARD"),
            col("earn_id")
        ).alias("movement_key"),

        date_format(
            col("earn_time"),
            "yyyyMMdd"
        ).cast("integer").alias("date_key"),

        upper(col("exchange")).alias(
            "exchange_key"
        ),

        upper(col("asset")).alias(
            "asset_key"
        ),

        lit("EARN_REWARD").alias(
            "movement_type"
        ),

        col("earn_time").alias(
            "movement_time"
        ),

        col("earn_wallet_change").alias(
            "quantity_change"
        )
    )
    .dropDuplicates([
        "movement_key"
    ])
)

print(
    "Mudrex Earn reward movement rows:",
    mudrex_earn_reward_movements_df.count()
)

display(
    mudrex_earn_reward_movements_df
    .groupBy("asset_key")
    .agg(
        count("*").alias("reward_count"),
        sum("quantity_change").alias("total_reward")
    )
    .orderBy("asset_key")
)

Mudrex Earn reward movement rows: 75


asset_key,reward_count,total_reward
BTC,1,1.0000000000E-8
ETH,30,0.000065060000000000
SOL,28,0.002280310000000000
SUI,6,0.000445460000000000
USDT,10,0.003568130000000000


In [0]:
gold_trades_df = spark.table(
    f"{catalog}.{gold_schema}.fact_trades"
)

trade_spent_df = (
    gold_trades_df
    .select(
        concat_ws(
            "|",
            col("trade_key"),
            lit("SPENT")
        ).alias("movement_key"),

        col("date_key"),
        col("exchange_key"),
        col("spent_asset_key").alias("asset_key"),
        lit("TRADE").alias("movement_type"),
        col("trade_time").alias("movement_time"),

        (-col("amount_spent")).alias(
            "quantity_change"
        )
    )
)

trade_received_df = (
    gold_trades_df
    .select(
        concat_ws(
            "|",
            col("trade_key"),
            lit("RECEIVED")
        ).alias("movement_key"),

        col("date_key"),
        col("exchange_key"),
        col("received_asset_key").alias("asset_key"),
        lit("TRADE").alias("movement_type"),
        col("trade_time").alias("movement_time"),

        col("amount_received").alias(
            "quantity_change"
        )
    )
)

trade_movements_df = (
    trade_spent_df
    .unionByName(
        trade_received_df
    )
)


display(
    trade_movements_df.orderBy(
        col("movement_time")
    )
)

movement_key,date_key,exchange_key,asset_key,movement_type,movement_time,quantity_change
Mudrex|Spot|BTC/USDT|17f3e7a64c003d8cc4133d24181a256672798cba304966d20b1bd92dd87c73dd|SPENT,20250530,MUDREX,USDT,TRADE,2025-05-30T12:11:25.000Z,-2.270399440000000000
Mudrex|Spot|BTC/USDT|17f3e7a64c003d8cc4133d24181a256672798cba304966d20b1bd92dd87c73dd|RECEIVED,20250530,MUDREX,BTC,TRADE,2025-05-30T12:11:25.000Z,0.000021540000000000
Mudrex|Spot|ETH/USDT|3bd6bff944cc9a3cd9037e18f7b84b2c8c71620ffe2988b840fc85a9bcb8f026|RECEIVED,20250530,MUDREX,ETH,TRADE,2025-05-30T12:12:06.000Z,0.000874200000000000
Mudrex|Spot|ETH/USDT|3bd6bff944cc9a3cd9037e18f7b84b2c8c71620ffe2988b840fc85a9bcb8f026|SPENT,20250530,MUDREX,USDT,TRADE,2025-05-30T12:12:06.000Z,-2.270399440000000000
Mudrex|Spot|SOL/USDT|9dba86cb8e3153b6361490713619869049f360a4f4146fd86750bd76d3f17574|SPENT,20250625,MUDREX,USDT,TRADE,2025-06-25T22:08:30.000Z,-1.133826870000000000
Mudrex|Spot|SOL/USDT|9dba86cb8e3153b6361490713619869049f360a4f4146fd86750bd76d3f17574|RECEIVED,20250625,MUDREX,SOL,TRADE,2025-06-25T22:08:30.000Z,0.007866470000000000
Mudrex|Spot|SOL/USDT|0ea7b3cd7aaa932d6d7577318cf8b36447e45eba91462dde108eb7412f78c370|SPENT,20250719,MUDREX,USDT,TRADE,2025-07-19T21:50:30.000Z,-2.240099000000000000
Mudrex|Spot|SOL/USDT|0ea7b3cd7aaa932d6d7577318cf8b36447e45eba91462dde108eb7412f78c370|RECEIVED,20250719,MUDREX,SOL,TRADE,2025-07-19T21:50:30.000Z,0.012730300000000000
Mudrex|Spot|ETH/USDT|66f2ec006ca83e6731925d9f28d8ffce771911afe90a44c60d2ae912f0617aec|RECEIVED,20250719,MUDREX,ETH,TRADE,2025-07-19T21:50:43.000Z,0.000627880000000000
Mudrex|Spot|ETH/USDT|66f2ec006ca83e6731925d9f28d8ffce771911afe90a44c60d2ae912f0617aec|SPENT,20250719,MUDREX,USDT,TRADE,2025-07-19T21:50:43.000Z,-2.240099000000000000


In [0]:
gold_p2p_df = spark.table(
    f"{catalog}.{gold_schema}.fact_p2p_transactions"
)

p2p_movements_df = (
    gold_p2p_df
    .filter(
        col("is_completed") == True
    )
    .select(
        concat_ws(
            "|",
            col("transaction_key"),
            lit("ASSET")
        ).alias("movement_key"),

        col("date_key"),
        col("exchange_key"),
        col("asset_key"),

        lit("P2P").alias("movement_type"),

        col("transaction_time").alias(
            "movement_time"
        ),

        when(
            upper(col("trade_type")) == "BUY",
            col("net_asset_amount")
        )
        .otherwise(
            -col("net_asset_amount")
        )
        .alias("quantity_change")
    )
)

print(
    "P2P movement rows:",
    p2p_movements_df.count()
)

display(
    p2p_movements_df.orderBy(
        col("movement_time")
    )
)

P2P movement rows: 13


movement_key,date_key,exchange_key,asset_key,movement_type,movement_time,quantity_change
BINANCE|P2P|22896212793618014208|ASSET,20260605,BINANCE,USDT,P2P,2026-06-05T13:56:25.112Z,9.560000000000000000
BINANCE|P2P|22896216150807388160|ASSET,20260605,BINANCE,USDT,P2P,2026-06-05T14:09:45.470Z,2.830000000000000000
BINANCE|P2P|22898272405559799808|ASSET,20260611,BINANCE,USDT,P2P,2026-06-11T06:20:34.842Z,5.010000000000000000
BINANCE|P2P|22898413188949139456|ASSET,20260611,BINANCE,USDT,P2P,2026-06-11T15:40:00.189Z,5.990000000000000000
BINANCE|P2P|22899404533812277248|ASSET,20260614,BINANCE,USDT,P2P,2026-06-14T09:19:15.226Z,6.020000000000000000
BINANCE|P2P|22906848402065932288|ASSET,20260704,BINANCE,USDT,P2P,2026-07-04T22:18:31.749Z,3.330000000000000000
BINANCE|P2P|22906848752644837376|ASSET,20260704,BINANCE,USDT,P2P,2026-07-04T22:19:55.318Z,2.850000000000000000
BINANCE|P2P|22907162745527037952|ASSET,20260705,BINANCE,USDT,P2P,2026-07-05T19:07:37.023Z,3.440000000000000000
BINANCE|P2P|22909209175754317824|ASSET,20260711,BINANCE,USDT,P2P,2026-07-11T10:39:24.047Z,3.360000000000000000
BINANCE|P2P|22910348341135761408|ASSET,20260714,BINANCE,USDT,P2P,2026-07-14T14:06:02.235Z,3.360000000000000000


In [0]:
# Binance deposits

binance_deposits_df = spark.table(
    f"{catalog}.{silver_schema}.binance_deposits"
)

binance_deposit_movements_df = (
    binance_deposits_df
    .filter(
        col("status_code").isin(1, 6)
    )
    .select(
        concat_ws(
            "|",
            upper(col("exchange")),
            lit("DEPOSIT"),
            col("deposit_id")
        ).alias("movement_key"),

        date_format(
            col("deposit_time"),
            "yyyyMMdd"
        ).cast("integer").alias("date_key"),

        upper(col("exchange"))
        .alias("exchange_key"),

        upper(col("asset"))
        .alias("asset_key"),

        lit("DEPOSIT")
        .alias("movement_type"),

        col("deposit_time")
        .alias("movement_time"),

        col("amount")
        .alias("quantity_change")
    )
)


# Giottus fiat deposits

giottus_fiat_deposits_df = spark.table(
    f"{catalog}.{silver_schema}.giottus_fiat_deposits"
)

giottus_fiat_deposit_movements_df = (
    giottus_fiat_deposits_df
    .filter(
        upper(col("status")) == "SUCCESS"
    )
    .select(
        concat_ws(
            "|",
            upper(col("exchange")),
            lit("FIAT_DEPOSIT"),
            col("deposit_id")
        ).alias("movement_key"),

        date_format(
            col("deposit_time"),
            "yyyyMMdd"
        ).cast("integer").alias("date_key"),

        upper(col("exchange"))
        .alias("exchange_key"),

        upper(col("asset"))
        .alias("asset_key"),

        lit("DEPOSIT")
        .alias("movement_type"),

        col("deposit_time")
        .alias("movement_time"),

        (
            col("amount")
            - coalesce(
                col("fee_amount"),
                lit(0).cast("decimal(38,18)")
            )
        ).alias("quantity_change")
    )
)


# Giottus crypto deposits

giottus_crypto_deposits_df = spark.table(
    f"{catalog}.{silver_schema}.giottus_crypto_deposits"
)

giottus_crypto_deposit_movements_df = (
    giottus_crypto_deposits_df
    .filter(
        upper(col("status")) == "SUCCESS"
    )
    .select(
        concat_ws(
            "|",
            upper(col("exchange")),
            lit("CRYPTO_DEPOSIT"),
            col("deposit_id")
        ).alias("movement_key"),

        date_format(
            col("deposit_time"),
            "yyyyMMdd"
        ).cast("integer").alias("date_key"),

        upper(col("exchange"))
        .alias("exchange_key"),

        upper(col("asset"))
        .alias("asset_key"),

        lit("DEPOSIT")
        .alias("movement_type"),

        col("deposit_time")
        .alias("movement_time"),

        (
            col("amount")
            - coalesce(
                col("fee_amount"),
                lit(0).cast("decimal(38,18)")
            )
        ).alias("quantity_change")
    )
)


# CoinDCX fiat deposits

coindcx_fiat_deposits_df = spark.table(
    f"{catalog}.{silver_schema}.coindcx_fiat_deposits"
)

coindcx_fiat_deposit_movements_df = (
    coindcx_fiat_deposits_df
    .filter(
        upper(col("status")) == "COMPLETED"
    )
    .select(
        concat_ws(
            "|",
            upper(col("exchange")),
            lit("FIAT_DEPOSIT"),
            col("deposit_id")
        ).alias("movement_key"),

        date_format(
            col("deposit_time"),
            "yyyyMMdd"
        ).cast("integer").alias("date_key"),

        upper(col("exchange"))
        .alias("exchange_key"),

        upper(col("asset"))
        .alias("asset_key"),

        lit("DEPOSIT")
        .alias("movement_type"),

        col("deposit_time")
        .alias("movement_time"),

        (
            col("amount")
            - coalesce(
                col("fee_amount"),
                lit(0).cast("decimal(38,18)")
            )
        ).alias("quantity_change")
    )
)


# CoinDCX crypto deposits

coindcx_crypto_deposits_df = spark.table(
    f"{catalog}.{silver_schema}.coindcx_crypto_deposits"
)

coindcx_crypto_deposit_movements_df = (
    coindcx_crypto_deposits_df
    .filter(
        upper(col("status")).isin(
            "SUCCESS",
            "COMPLETED"
        )
    )
    .select(
        concat_ws(
            "|",
            upper(col("exchange")),
            lit("CRYPTO_DEPOSIT"),
            col("deposit_id")
        ).alias("movement_key"),

        date_format(
            col("deposit_time"),
            "yyyyMMdd"
        ).cast("integer").alias("date_key"),

        upper(col("exchange"))
        .alias("exchange_key"),

        upper(col("asset"))
        .alias("asset_key"),

        lit("DEPOSIT")
        .alias("movement_type"),

        col("deposit_time")
        .alias("movement_time"),

        (
            col("amount")
            - coalesce(
                col("fee_amount"),
                lit(0).cast("decimal(38,18)")
            )
        ).alias("quantity_change")
    )
)


# Combine all deposits

deposit_movements_df = (
    binance_deposit_movements_df
    .unionByName(
        giottus_fiat_deposit_movements_df
    )
    .unionByName(
        giottus_crypto_deposit_movements_df
    )
    .unionByName(
        coindcx_fiat_deposit_movements_df
    )
    .unionByName(
        coindcx_crypto_deposit_movements_df
    )
)

print(
    "Deposit movement rows:",
    deposit_movements_df.count()
)

display(
    deposit_movements_df
    .groupBy("exchange_key")
    .count()
    .orderBy("exchange_key")
)

display(
    deposit_movements_df
    .orderBy("movement_time")
)

Deposit movement rows: 3


exchange_key,count
COINDCX,1
GIOTTUS,2


movement_key,date_key,exchange_key,asset_key,movement_type,movement_time,quantity_change
COINDCX|FIAT_DEPOSIT|422144727195,20260314,COINDCX,INR,DEPOSIT,2026-03-14T15:36:00.000Z,1000.00000000000000000
GIOTTUS|FIAT_DEPOSIT|792452268163,20260318,GIOTTUS,INR,DEPOSIT,2026-03-18T20:26:49.000Z,100.00000000000000000
GIOTTUS|FIAT_DEPOSIT|391465032548,20260609,GIOTTUS,INR,DEPOSIT,2026-06-09T01:32:53.000Z,100.00000000000000000


In [0]:
# Binance withdrawals

binance_withdrawals_df = spark.table(
    f"{catalog}.{silver_schema}.binance_withdrawals"
)

binance_withdrawal_movements_df = (
    binance_withdrawals_df
    .filter(
        col("status_code") == 6
    )
    .withColumn(
        "movement_timestamp",
        coalesce(
            col("completed_at"),
            col("requested_at")
        )
    )
    .select(
        concat_ws(
            "|",
            upper(col("exchange")),
            lit("WITHDRAWAL"),
            col("withdrawal_id")
        ).alias("movement_key"),

        date_format(
            col("movement_timestamp"),
            "yyyyMMdd"
        ).cast("integer").alias("date_key"),

        upper(col("exchange"))
        .alias("exchange_key"),

        upper(col("asset"))
        .alias("asset_key"),

        lit("WITHDRAWAL")
        .alias("movement_type"),

        col("movement_timestamp")
        .alias("movement_time"),

        (
            -(
                col("amount")
                + coalesce(
                    col("transaction_fee"),
                    lit(0).cast("decimal(38,18)")
                )
            )
        ).alias("quantity_change")
    )
)


# Giottus fiat withdrawals

giottus_fiat_withdrawals_df = spark.table(
    f"{catalog}.{silver_schema}.giottus_fiat_withdrawals"
)

giottus_fiat_withdrawal_movements_df = (
    giottus_fiat_withdrawals_df
    .filter(
        upper(col("status")) == "SUCCESS"
    )
    .select(
        concat_ws(
            "|",
            upper(col("exchange")),
            lit("FIAT_WITHDRAWAL"),
            col("withdrawal_id")
        ).alias("movement_key"),

        date_format(
            col("withdrawal_time"),
            "yyyyMMdd"
        ).cast("integer").alias("date_key"),

        upper(col("exchange"))
        .alias("exchange_key"),

        upper(col("asset"))
        .alias("asset_key"),

        lit("WITHDRAWAL")
        .alias("movement_type"),

        col("withdrawal_time")
        .alias("movement_time"),

        (
            -(
                col("amount")
                + coalesce(
                    col("fee_amount"),
                    lit(0).cast("decimal(38,18)")
                )
            )
        ).alias("quantity_change")
    )
)


# Giottus crypto withdrawals

giottus_crypto_withdrawals_df = spark.table(
    f"{catalog}.{silver_schema}.giottus_crypto_withdrawals"
)

giottus_crypto_withdrawal_movements_df = (
    giottus_crypto_withdrawals_df
    .filter(
        upper(col("status")) == "SUCCESS"
    )
    .select(
        concat_ws(
            "|",
            upper(col("exchange")),
            lit("CRYPTO_WITHDRAWAL"),
            col("withdrawal_id")
        ).alias("movement_key"),

        date_format(
            col("withdrawal_time"),
            "yyyyMMdd"
        ).cast("integer").alias("date_key"),

        upper(col("exchange"))
        .alias("exchange_key"),

        upper(col("asset"))
        .alias("asset_key"),

        lit("WITHDRAWAL")
        .alias("movement_type"),

        col("withdrawal_time")
        .alias("movement_time"),

        (
            -(
                col("amount")
                + coalesce(
                    col("transaction_fee"),
                    lit(0).cast("decimal(38,18)")
                )
            )
        ).alias("quantity_change")
    )
)


# CoinDCX fiat withdrawals

coindcx_fiat_withdrawals_df = spark.table(
    f"{catalog}.{silver_schema}.coindcx_fiat_withdrawals"
)

coindcx_fiat_withdrawal_movements_df = (
    coindcx_fiat_withdrawals_df
    .filter(
        upper(col("status")).isin(
            "SUCCESS",
            "COMPLETED"
        )
    )
    .select(
        concat_ws(
            "|",
            upper(col("exchange")),
            lit("FIAT_WITHDRAWAL"),
            col("withdrawal_id")
        ).alias("movement_key"),

        date_format(
            col("withdrawal_time"),
            "yyyyMMdd"
        ).cast("integer").alias("date_key"),

        upper(col("exchange"))
        .alias("exchange_key"),

        upper(col("asset"))
        .alias("asset_key"),

        lit("WITHDRAWAL")
        .alias("movement_type"),

        col("withdrawal_time")
        .alias("movement_time"),

        (
            -(
                col("amount")
                + coalesce(
                    col("fee_amount"),
                    lit(0).cast("decimal(38,18)")
                )
            )
        ).alias("quantity_change")
    )
)


# CoinDCX crypto withdrawals

coindcx_crypto_withdrawals_df = spark.table(
    f"{catalog}.{silver_schema}.coindcx_crypto_withdrawals"
)

coindcx_crypto_withdrawal_movements_df = (
    coindcx_crypto_withdrawals_df
    .filter(
        upper(col("status")).isin(
            "SUCCESS",
            "COMPLETED"
        )
    )
    .select(
        concat_ws(
            "|",
            upper(col("exchange")),
            lit("CRYPTO_WITHDRAWAL"),
            col("withdrawal_id")
        ).alias("movement_key"),

        date_format(
            col("withdrawal_time"),
            "yyyyMMdd"
        ).cast("integer").alias("date_key"),

        upper(col("exchange"))
        .alias("exchange_key"),

        upper(col("asset"))
        .alias("asset_key"),

        lit("WITHDRAWAL")
        .alias("movement_type"),

        col("withdrawal_time")
        .alias("movement_time"),

        (
            -(
                col("amount")
                + coalesce(
                    col("transaction_fee"),
                    lit(0).cast("decimal(38,18)")
                )
            )
        ).alias("quantity_change")
    )
)


# Combine all withdrawals

withdrawal_movements_df = (
    binance_withdrawal_movements_df
    .unionByName(
        giottus_fiat_withdrawal_movements_df
    )
    .unionByName(
        giottus_crypto_withdrawal_movements_df
    )
    .unionByName(
        coindcx_fiat_withdrawal_movements_df
    )
    .unionByName(
        coindcx_crypto_withdrawal_movements_df
    )
)

print(
    "Withdrawal movement rows:",
    withdrawal_movements_df.count()
)

display(
    withdrawal_movements_df
    .groupBy("exchange_key")
    .count()
    .orderBy("exchange_key")
)

display(
    withdrawal_movements_df
    .orderBy("movement_time")
)

Withdrawal movement rows: 2


exchange_key,count
BINANCE,1
GIOTTUS,1


movement_key,date_key,exchange_key,asset_key,movement_type,movement_time,quantity_change
BINANCE|WITHDRAWAL|3f2234d4647746d4ac478d1e8d792855,20260313,BINANCE,SOL,WITHDRAWAL,2026-03-13T06:41:06.000Z,-0.10765482000000000
GIOTTUS|CRYPTO_WITHDRAWAL|6686245,20260629,GIOTTUS,SOL,WITHDRAWAL,2026-06-29T03:29:59.000Z,-0.02658800000000000


In [0]:
silver_transfers_df = spark.table(
    f"{catalog}.{silver_schema}.binance_transfers"
)

transfer_movements_df = (
    silver_transfers_df
    .filter(
        upper(col("status")).isin(
            "CONFIRMED",
            "SUCCESS"
        )
    )
    .withColumn(
        "transfer_quantity_change",

        when(
            upper(
                col("requested_transfer_type")
            ).startswith("MAIN_"),

            -col("amount")
        )
        .when(
            upper(
                col("requested_transfer_type")
            ).endswith("_MAIN"),

            col("amount")
        )
        .otherwise(
            lit(0).cast("decimal(38,18)")
        )
    )
    .filter(
        col("transfer_quantity_change") != 0
    )
    .select(
        concat_ws(
            "|",
            upper(col("exchange")),
            lit("TRANSFER"),
            col("transfer_id")
        ).alias("movement_key"),

        date_format(
            col("transfer_time"),
            "yyyyMMdd"
        )
        .cast("integer")
        .alias("date_key"),

        upper(col("exchange")).alias(
            "exchange_key"
        ),

        upper(col("asset")).alias(
            "asset_key"
        ),

        lit("TRANSFER").alias(
            "movement_type"
        ),

        col("transfer_time").alias(
            "movement_time"
        ),

        col("transfer_quantity_change").alias(
            "quantity_change"
        )
    )
)

print(
    "Transfer movement rows:",
    transfer_movements_df.count()
)

display(transfer_movements_df)

Transfer movement rows: 0


movement_key,date_key,exchange_key,asset_key,movement_type,movement_time,quantity_change


In [0]:
trade_fee_movements_df = (
    gold_trades_df
    .filter(
        col("fee_amount").isNotNull()
        & (col("fee_amount") > 0)
    )
    .select(
        concat_ws(
            "|",
            col("trade_key"),
            lit("FEE")
        ).alias("movement_key"),

        col("date_key"),
        col("exchange_key"),

        upper(
            col("fee_asset")
        ).alias("asset_key"),

        lit("TRADE_FEE").alias(
            "movement_type"
        ),

        col("trade_time").alias(
            "movement_time"
        ),

        (
            -col("fee_amount")
        ).alias(
            "quantity_change"
        )
    )
)


print(
    "Trade fee movement rows:",
    trade_fee_movements_df.count()
)


display(
    trade_fee_movements_df.orderBy(
        col("movement_time")
    )
)

Trade fee movement rows: 16


movement_key,date_key,exchange_key,asset_key,movement_type,movement_time,quantity_change
CoinDCX|Instant|SOL/INR|292a7605-4955-4d01-b296-41d32c14920e|FEE,20260314,COINDCX,INR,TRADE_FEE,2026-03-14T15:50:21.000Z,-5.865190000000000000
Giottus|EBES|SOL/INR|2858754|FEE,20260319,GIOTTUS,SOL,TRADE_FEE,2026-03-19T13:01:22.000Z,-0.000046400000000000
Binance|Spot|BTCUSDT|6369541361|FEE,20260605,BINANCE,BTC,TRADE_FEE,2026-06-05T14:13:30.940Z,-8.0000000000E-8
Binance|Spot|SOLUSDT|1972197389|FEE,20260605,BINANCE,SOL,TRADE_FEE,2026-06-05T14:14:53.697Z,-0.000027000000000000
Binance|Spot|SOLUSDT|1972197391|FEE,20260605,BINANCE,SOL,TRADE_FEE,2026-06-05T14:14:53.697Z,-0.000009000000000000
Binance|Spot|SOLUSDT|1972197390|FEE,20260605,BINANCE,SOL,TRADE_FEE,2026-06-05T14:14:53.697Z,-0.000079000000000000
Giottus|Spot|SOL/INR|228465533|FEE,20260609,GIOTTUS,SOL,TRADE_FEE,2026-06-09T21:54:49.000Z,-0.000060000000000000
Binance|Spot|ETHUSDT|4121758522|FEE,20260611,BINANCE,ETH,TRADE_FEE,2026-06-11T06:28:07.152Z,-0.000003000000000000
Binance|Spot|SUIUSDT|523831127|FEE,20260611,BINANCE,SUI,TRADE_FEE,2026-06-11T15:50:18.493Z,-0.008100000000000000
Binance|Spot|LINKUSDT|472919659|FEE,20260614,BINANCE,LINK,TRADE_FEE,2026-06-14T09:38:33.928Z,-0.000640000000000000


In [0]:
all_asset_movements_df = (
    trade_movements_df

    .unionByName(
        p2p_movements_df
    )

    .unionByName(
        deposit_movements_df
    )

    .unionByName(
        withdrawal_movements_df
    )

    .unionByName(
        transfer_movements_df
    )

    .unionByName(
        trade_fee_movements_df
    )

    .unionByName(
        mudrex_deposit_movements_df
    )

    .unionByName(
        mudrex_withdrawal_movements_df
    )

    .unionByName(
        mudrex_earn_reward_movements_df
    )
)


fact_asset_movements_df = (
    all_asset_movements_df
    .filter(
        col("movement_key").isNotNull()
        & col("asset_key").isNotNull()
        & col("movement_time").isNotNull()
        & col("quantity_change").isNotNull()
    )
    .withColumn(
        "quantity_change",
        col("quantity_change")
        .cast("decimal(38,18)")
    )
    .dropDuplicates([
        "movement_key"
    ])
    .withColumn(
        "gold_processed_at",
        current_timestamp()
    )
)


print(
    "Total asset movement rows:",
    fact_asset_movements_df.count()
)


display(
    fact_asset_movements_df
    .groupBy(
        "exchange_key",
        "movement_type"
    )
    .count()
    .orderBy(
        "exchange_key",
        "movement_type"
    )
)


display(
    fact_asset_movements_df.orderBy(
        col("movement_time")
    )
)

Total asset movement rows: 236


exchange_key,movement_type,count
BINANCE,P2P,13
BINANCE,TRADE,32
BINANCE,TRADE_FEE,13
BINANCE,WITHDRAWAL,1
COINDCX,DEPOSIT,1
COINDCX,TRADE,2
COINDCX,TRADE_FEE,1
GIOTTUS,DEPOSIT,2
GIOTTUS,TRADE,4
GIOTTUS,TRADE_FEE,2


movement_key,date_key,exchange_key,asset_key,movement_type,movement_time,quantity_change,gold_processed_at
MUDREX|DEPOSIT|955bce71-177c-43d1-ba3f-2d6e72ff046c,20250530,MUDREX,USDT,DEPOSIT,2025-05-30T11:34:12.000Z,5.690224190000000000,2026-10-02T16:33:19.957Z
Mudrex|Spot|BTC/USDT|17f3e7a64c003d8cc4133d24181a256672798cba304966d20b1bd92dd87c73dd|SPENT,20250530,MUDREX,USDT,TRADE,2025-05-30T12:11:25.000Z,-2.270399440000000000,2026-10-02T16:33:19.957Z
Mudrex|Spot|BTC/USDT|17f3e7a64c003d8cc4133d24181a256672798cba304966d20b1bd92dd87c73dd|RECEIVED,20250530,MUDREX,BTC,TRADE,2025-05-30T12:11:25.000Z,0.000021540000000000,2026-10-02T16:33:19.957Z
Mudrex|Spot|ETH/USDT|3bd6bff944cc9a3cd9037e18f7b84b2c8c71620ffe2988b840fc85a9bcb8f026|SPENT,20250530,MUDREX,USDT,TRADE,2025-05-30T12:12:06.000Z,-2.270399440000000000,2026-10-02T16:33:19.957Z
Mudrex|Spot|ETH/USDT|3bd6bff944cc9a3cd9037e18f7b84b2c8c71620ffe2988b840fc85a9bcb8f026|RECEIVED,20250530,MUDREX,ETH,TRADE,2025-05-30T12:12:06.000Z,0.000874200000000000,2026-10-02T16:33:19.957Z
MUDREX|EARN_REWARD|81f57de4b291a4f6e788742ae8c8f53298c759b9e587e62726343bd83165d96e,20250602,MUDREX,ETH,EARN_REWARD,2025-06-02T09:00:06.000Z,2.0000000000E-8,2026-10-02T16:33:19.957Z
MUDREX|EARN_REWARD|8f3911e4e84a8729d2db8ec5b409aca4d5645f5519135c833371e2eef194791b,20250603,MUDREX,USDT,EARN_REWARD,2025-06-03T10:03:32.000Z,0.000138040000000000,2026-10-02T16:33:19.957Z
MUDREX|EARN_REWARD|7c762f35de7835b4ee4baec0c46b5e1afb06eb23ee9864a45ef49f8932d12d6f,20250603,MUDREX,ETH,EARN_REWARD,2025-06-03T10:03:51.000Z,3.0000000000E-8,2026-10-02T16:33:19.957Z
MUDREX|EARN_REWARD|30980d9583fb999d28751fd333ad620b9906c924f8f7adce06f746f7d3a9001d,20250604,MUDREX,USDT,EARN_REWARD,2025-06-04T09:13:28.000Z,0.000138040000000000,2026-10-02T16:33:19.957Z
MUDREX|EARN_REWARD|6ea9317d64001f36a6524a5b397cc6163d07628f243cd34aa30076a13683651b,20250604,MUDREX,ETH,EARN_REWARD,2025-06-04T09:13:51.000Z,3.0000000000E-8,2026-10-02T16:33:19.957Z


In [0]:
(
    fact_asset_movements_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{gold_schema}.fact_asset_movements"
    )
)

saved_movements_df = spark.table(
    f"{catalog}.{gold_schema}.fact_asset_movements"
)

print(
    "Saved asset movement rows:",
    saved_movements_df.count()
)

display(
    saved_movements_df
    .groupBy("movement_type")
    .count()
    .orderBy("movement_type")
)

Saved asset movement rows: 236


movement_type,count
DEPOSIT,17
EARN_REWARD,75
P2P,13
TRADE,112
TRADE_FEE,16
WITHDRAWAL,3


####Fact daily balances

In [0]:
mudrex_current_balances_df = (
    saved_movements_df
    .filter(
        col("exchange_key") == "MUDREX"
    )
    .groupBy(
        "exchange_key",
        "asset_key"
    )
    .agg(
        sum("quantity_change").alias(
            "current_balance"
        )
    )
)

print(
    "Calculated Mudrex balance rows:",
    mudrex_current_balances_df.count()
)

display(
    mudrex_current_balances_df
    .orderBy("asset_key")
)

Calculated Mudrex balance rows: 8


exchange_key,asset_key,current_balance
MUDREX,BTC,0.000517887616099000
MUDREX,ETH,0.022118925435541000
MUDREX,INR,0E-18
MUDREX,LINK,1.110548702356832000
MUDREX,ONDO,38.452899654224275000
MUDREX,SOL,0.446967153609897000
MUDREX,SUI,13.936812729018684000
MUDREX,USDT,0.750902305217556000


In [0]:
balance_snapshots_df = spark.table(
    f"{catalog}.{gold_schema}.fact_balance_snapshots"
)


# Find the latest snapshot time for each wallet

latest_snapshot_times_df = (
    balance_snapshots_df
    .groupBy(
        "exchange_key",
        "wallet_key"
    )
    .agg(
        max("snapshot_time").alias(
            "latest_snapshot_time"
        )
    )
)


# Get every asset from each latest wallet snapshot

latest_wallet_balances_df = (
    balance_snapshots_df.alias("balances")
    .join(
        latest_snapshot_times_df.alias("latest"),
        (
            (
                col("balances.exchange_key")
                == col("latest.exchange_key")
            )
            & (
                col("balances.wallet_key")
                == col("latest.wallet_key")
            )
            & (
                col("balances.snapshot_time")
                == col("latest.latest_snapshot_time")
            )
        ),
        "inner"
    )
    .select(
        col("balances.exchange_key"),
        col("balances.asset_key").alias("asset"),
        col("balances.total_balance")
    )
    .filter(
        col("total_balance") > 0
    )
)


# Combine wallets belonging to the same exchange

latest_total_balances_df = (
    latest_wallet_balances_df
    .groupBy(
        "exchange_key",
        "asset"
    )
    .agg(
        sum("total_balance").alias(
            "current_balance"
        )
    )
)


print(
    "Latest total balance rows:",
    latest_total_balances_df.count()
)

display(
    latest_total_balances_df.orderBy(
        "exchange_key",
        "asset"
    )
)

Latest total balance rows: 10


exchange_key,asset,current_balance
BINANCE,BTC,0.000289710000000000
BINANCE,ETH,0.006993000000000000
BINANCE,LINK,0.759240000000000000
BINANCE,ONDO,18.281700000000000000
BINANCE,SOL,0.114885000000000000
BINANCE,SUI,8.091900000000000000
BINANCE,USDT,0.430995000000000000
COINDCX,INR,0.034810000000000000
COINDCX,SOL,0.121948825900600000
GIOTTUS,INR,0.004980010000000000


In [0]:
mudrex_current_balances_df = (
    spark.table(
        f"{catalog}.{gold_schema}.fact_asset_movements"
    )
    .filter(
        col("exchange_key") == "MUDREX"
    )
    .groupBy(
        "exchange_key",
        "asset_key"
    )
    .agg(
        sum("quantity_change").alias(
            "current_balance"
        )
    )
)


mudrex_current_for_daily_df = (
    mudrex_current_balances_df
    .select(
        col("exchange_key"),
        col("asset_key").alias("asset"),
        col("current_balance")
    )
)


all_current_balances_df = (
    latest_total_balances_df
    .unionByName(
        mudrex_current_for_daily_df
    )
    .dropDuplicates([
        "exchange_key",
        "asset"
    ])
)


print(
    "All current balance rows:",
    all_current_balances_df.count()
)


display(
    all_current_balances_df
    .orderBy(
        "exchange_key",
        "asset"
    )
)

All current balance rows: 18


exchange_key,asset,current_balance
BINANCE,BTC,0.000289710000000000
BINANCE,ETH,0.006993000000000000
BINANCE,LINK,0.759240000000000000
BINANCE,ONDO,18.281700000000000000
BINANCE,SOL,0.114885000000000000
BINANCE,SUI,8.091900000000000000
BINANCE,USDT,0.430995000000000000
COINDCX,INR,0.034810000000000000
COINDCX,SOL,0.121948825900600000
GIOTTUS,INR,0.004980010000000000


In [0]:
movements_df = spark.table(
    f"{catalog}.{gold_schema}.fact_asset_movements"
)


daily_movements_df = (
    movements_df
    .withColumn(
        "movement_date",
        to_date(col("movement_time"))
    )
    .groupBy(
        "movement_date",
        "exchange_key",
        "asset_key"
    )
    .agg(
        sum("quantity_change").alias(
            "daily_change"
        )
    )
)


print(
    "Daily movement rows:",
    daily_movements_df.count()
)


display(
    daily_movements_df.orderBy(
        "movement_date",
        "exchange_key",
        "asset_key"
    )
)

Daily movement rows: 144


movement_date,exchange_key,asset_key,daily_change
2025-05-30,MUDREX,BTC,0.000021540000000000
2025-05-30,MUDREX,ETH,0.000874200000000000
2025-05-30,MUDREX,USDT,1.149425310000000000
2025-06-02,MUDREX,ETH,2.0000000000E-8
2025-06-03,MUDREX,ETH,3.0000000000E-8
2025-06-03,MUDREX,USDT,0.000138040000000000
2025-06-04,MUDREX,ETH,3.0000000000E-8
2025-06-04,MUDREX,USDT,0.000138040000000000
2025-06-05,MUDREX,ETH,3.0000000000E-8
2025-06-05,MUDREX,USDT,0.000138040000000000


In [0]:
date_df = spark.table(
    f"{catalog}.{gold_schema}.dim_date"
)
date_df.printSchema()
print("total date rows: ", date_df.count())

root
 |-- date_key: integer (nullable = true)
 |-- full_date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- month_number: integer (nullable = true)
 |-- month_name: string (nullable = true)
 |-- day_of_month: integer (nullable = true)
 |-- day_name: string (nullable = true)
 |-- week_of_year: integer (nullable = true)
 |-- is_weekend: boolean (nullable = true)
 |-- gold_processed_at: timestamp (nullable = true)

total date rows:  492


In [0]:
date_range_df = (
    date_df
    .agg(
        min("full_date").alias("start_date"),
        max("full_date").alias("end_date")
    )
)

display(date_range_df)

start_date,end_date
2025-05-29,2026-10-02


In [0]:
assets_df = spark.table(
    f"{catalog}.{gold_schema}.dim_asset"
)
assets_df.printSchema()
print("Asset rows: ",assets_df.count())

root
 |-- asset_key: string (nullable = true)
 |-- asset_symbol: string (nullable = true)
 |-- gold_processed_at: timestamp (nullable = true)

Asset rows:  8


In [0]:
balance_dates_df = (
    date_df
    .select(
        col("full_date").alias(
            "balance_date"
        )
    )
)


current_exchange_assets_df = (
    all_current_balances_df
    .select(
        col("exchange_key"),
        upper(col("asset")).alias(
            "asset_key"
        )
    )
)


movement_exchange_assets_df = (
    daily_movements_df
    .select(
        "exchange_key",
        "asset_key"
    )
)


balance_exchange_assets_df = (
    current_exchange_assets_df
    .unionByName(
        movement_exchange_assets_df
    )
    .dropDuplicates([
        "exchange_key",
        "asset_key"
    ])
)


balance_calendar_df = (
    balance_dates_df
    .crossJoin(
        balance_exchange_assets_df
    )
)


print(
    "Balance calendar rows:",
    balance_calendar_df.count()
)


display(
    balance_calendar_df.orderBy(
        "exchange_key",
        "asset_key",
        "balance_date"
    )
)

Balance calendar rows: 9348


balance_date,exchange_key,asset_key
2025-05-29,BINANCE,BTC
2025-05-30,BINANCE,BTC
2025-05-31,BINANCE,BTC
2025-06-01,BINANCE,BTC
2025-06-02,BINANCE,BTC
2025-06-03,BINANCE,BTC
2025-06-04,BINANCE,BTC
2025-06-05,BINANCE,BTC
2025-06-06,BINANCE,BTC
2025-06-07,BINANCE,BTC


In [0]:
daily_changes_df = (
    daily_movements_df
    .withColumnRenamed(
        "movement_date",
        "balance_date"
    )
)


balance_activity_df = (
    balance_calendar_df
    .join(
        daily_changes_df,
        [
            "balance_date",
            "exchange_key",
            "asset_key"
        ],
        "left"
    )
    .fillna(
        0,
        subset=["daily_change"]
    )
)


print(
    "Balance activity rows:",
    balance_activity_df.count()
)

Balance activity rows: 9348


In [0]:
current_balances_df = (
    all_current_balances_df
    .select(
        col("exchange_key"),

        upper(col("asset")).alias(
            "asset_key"
        ),

        col("current_balance")
    )
)


balance_base_df = (
    balance_activity_df
    .join(
        current_balances_df,
        [
            "exchange_key",
            "asset_key"
        ],
        "left"
    )
    .fillna(
        0,
        subset=["current_balance"]
    )
)


print(
    "Balance base rows:",
    balance_base_df.count()
)

Balance base rows: 9348


In [0]:
from pyspark.sql.window import Window


future_movements_window = (
    Window
    .partitionBy(
        "exchange_key",
        "asset_key"
    )
    .orderBy(
        "balance_date"
    )
    .rowsBetween(
        1,
        Window.unboundedFollowing
    )
)


balance_with_future_df = (
    balance_base_df
    .withColumn(
        "movements_after_date",
        coalesce(
            sum("daily_change").over(
                future_movements_window
            ),
            lit(0).cast(
                "decimal(38,18)"
            )
        )
    )
)


display(
    balance_with_future_df
    .select(
        "balance_date",
        "exchange_key",
        "asset_key",
        "daily_change",
        "current_balance",
        "movements_after_date"
    )
    .orderBy(
        "exchange_key",
        "asset_key",
        "balance_date"
    )
)

balance_date,exchange_key,asset_key,daily_change,current_balance,movements_after_date
2025-05-29,BINANCE,BTC,0E-18,0.000289710000000000,0.000289710000000000
2025-05-30,BINANCE,BTC,0E-18,0.000289710000000000,0.000289710000000000
2025-05-31,BINANCE,BTC,0E-18,0.000289710000000000,0.000289710000000000
2025-06-01,BINANCE,BTC,0E-18,0.000289710000000000,0.000289710000000000
2025-06-02,BINANCE,BTC,0E-18,0.000289710000000000,0.000289710000000000
2025-06-03,BINANCE,BTC,0E-18,0.000289710000000000,0.000289710000000000
2025-06-04,BINANCE,BTC,0E-18,0.000289710000000000,0.000289710000000000
2025-06-05,BINANCE,BTC,0E-18,0.000289710000000000,0.000289710000000000
2025-06-06,BINANCE,BTC,0E-18,0.000289710000000000,0.000289710000000000
2025-06-07,BINANCE,BTC,0E-18,0.000289710000000000,0.000289710000000000


In [0]:
calculated_balances_df = (
    balance_with_future_df
    .withColumn(
        "balance_amount",
        (
            col("current_balance")
            - col("movements_after_date")
        )
    )
)

print(
    "Calculated rows:",
    calculated_balances_df.count()
)

print(
    "Negative balance rows:",
    calculated_balances_df
    .filter(col("balance_amount") < 0)
    .count()
)
display(
    calculated_balances_df
    .orderBy(
        col("balance_date").asc()
    )
)

Calculated rows: 9348
Negative balance rows: 0


exchange_key,asset_key,balance_date,daily_change,current_balance,movements_after_date,balance_amount
BINANCE,ONDO,2025-05-29,0E-18,18.281700000000000000,18.281700000000000000,0E-17
BINANCE,USDT,2025-05-29,0E-18,0.430995000000000000,-9.099005580000000000,9.53000058000000000
BINANCE,SOL,2025-05-29,0E-18,0.114885000000000000,0.114885000000000000,0E-17
GIOTTUS,SOL,2025-05-29,0E-18,0E-18,-0.000036400000000000,0.00003640000000000
MUDREX,USDT,2025-05-29,0E-18,0.750902305217556000,0.750902305217556000,0E-17
MUDREX,SOL,2025-05-29,0E-18,0.446967153609897000,0.446967153609897000,0E-17
MUDREX,ETH,2025-05-29,0E-18,0.022118925435541000,0.022118925435541000,0E-17
BINANCE,ETH,2025-05-29,0E-18,0.006993000000000000,0.006993000000000000,0E-17
BINANCE,SUI,2025-05-29,0E-18,8.091900000000000000,8.091900000000000000,0E-17
MUDREX,INR,2025-05-29,0E-18,0E-18,0E-18,0E-17


In [0]:
fact_daily_balances_df = (
    calculated_balances_df
    .withColumn(
        "date_key",
        date_format(
            col("balance_date"),
            "yyyyMMdd"
        ).cast("integer")
    )
    .withColumn(
        "balance_key",
        concat_ws(
            "|",
            col("exchange_key"),
            col("asset_key"),
            col("date_key")
        )
    )
    .withColumn(
        "gold_processed_at",
        current_timestamp()
    )
    .select(
        "balance_key",
        "date_key",
        "balance_date",
        "exchange_key",
        "asset_key",
        "balance_amount",
        "daily_change",
        "gold_processed_at"
    )
)


fact_daily_balances_df.printSchema()

root
 |-- balance_key: string (nullable = false)
 |-- date_key: integer (nullable = true)
 |-- balance_date: date (nullable = true)
 |-- exchange_key: string (nullable = true)
 |-- asset_key: string (nullable = true)
 |-- balance_amount: decimal(38,17) (nullable = false)
 |-- daily_change: decimal(38,18) (nullable = false)
 |-- gold_processed_at: timestamp (nullable = false)



In [0]:
duplicate_keys = (
    fact_daily_balances_df
    .groupBy("balance_key")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Duplicate balance keys:", duplicate_keys)

Duplicate balance keys: 0


In [0]:
(
    fact_daily_balances_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{gold_schema}.fact_daily_balances"
    )
)

saved_daily_balances_df = spark.table(
    f"{catalog}.{gold_schema}.fact_daily_balances"
)

print(
    "Saved daily balance rows:",
    saved_daily_balances_df.count()
)

Saved daily balance rows: 9348


####Fact daily prices

In [0]:
silver_price_history = (
    spark.table(
        f"{catalog}.{silver_schema}.binance_price_history"
    )
)

print("silver price history rows: ",silver_price_history.count())

silver_price_history.printSchema()

display(
    silver_price_history
    .orderBy(
        col("open_time").desc()
    )
    .limit(5)
)


silver price history rows:  2952
root
 |-- symbol: string (nullable = true)
 |-- asset: string (nullable = true)
 |-- open_time: timestamp (nullable = true)
 |-- close_time: timestamp (nullable = true)
 |-- open_price: decimal(38,18) (nullable = true)
 |-- high_price: decimal(38,18) (nullable = true)
 |-- low_price: decimal(38,18) (nullable = true)
 |-- close_price: decimal(38,18) (nullable = true)
 |-- volume: decimal(38,18) (nullable = true)
 |-- quote_asset_volume: decimal(38,18) (nullable = true)
 |-- trade_count: long (nullable = true)
 |-- taker_buy_base_volume: decimal(38,18) (nullable = true)
 |-- taker_buy_quote_volume: decimal(38,18) (nullable = true)
 |-- exchange: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- source_file_modified_at: timestamp (nullable = true)
 |-- source_file_saved_at: timestamp (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- silver_processed_at: timestamp (nullable = true)
 |-- is_closed: boolean (n

symbol,asset,open_time,close_time,open_price,high_price,low_price,close_price,volume,quote_asset_volume,trade_count,taker_buy_base_volume,taker_buy_quote_volume,exchange,_source_file,source_file_modified_at,source_file_saved_at,bronze_ingested_at,silver_processed_at,is_closed,price_change,price_change_pct
ETHUSDT,ETH,2026-10-02T00:00:00.000Z,2026-10-02T23:59:59.999Z,2706.430000000000000000,2777.330000000000000000,2696.560000000000000000,2738.720000000000000000,226154.564500000000000000,619661901.583315000000000000,2610080,116873.930800000000000000,320135257.194681000000000000,Binance,s3://l1l2bucket/data/raw/binance/price_history/ETHUSDT/price_history_20261002T141804_360765Z_207c580a.json,2026-10-02T14:18:03.000Z,2026-10-02T14:18:04.360Z,2026-10-02T14:23:21.544Z,2026-10-02T14:45:48.883Z,false,32.290000000000000000,1.193100000000000000
SUIUSDT,SUI,2026-10-02T00:00:00.000Z,2026-10-02T23:59:59.999Z,1.177200000000000000,1.218600000000000000,1.163000000000000000,1.186700000000000000,54223248.200000000000000000,64482395.634370000000000000,462689,28466524.800000000000000000,33868676.946220000000000000,Binance,s3://l1l2bucket/data/raw/binance/price_history/SUIUSDT/price_history_20261002T141805_284183Z_b09085f0.json,2026-10-02T14:18:04.000Z,2026-10-02T14:18:05.284Z,2026-10-02T14:23:21.544Z,2026-10-02T14:45:48.883Z,false,0.009500000000000000,0.807000000000000000
ONDOUSDT,ONDO,2026-10-02T00:00:00.000Z,2026-10-02T23:59:59.999Z,0.497200000000000000,0.516200000000000000,0.489600000000000000,0.506500000000000000,34221727.900000000000000000,17316882.661230000000000000,83798,16609635.500000000000000000,8403755.947520000000000000,Binance,s3://l1l2bucket/data/raw/binance/price_history/ONDOUSDT/price_history_20261002T141806_205850Z_0b2ff977.json,2026-10-02T14:18:05.000Z,2026-10-02T14:18:06.205Z,2026-10-02T14:23:21.544Z,2026-10-02T14:45:48.883Z,false,0.009300000000000000,1.870500000000000000
BTCUSDT,BTC,2026-10-02T00:00:00.000Z,2026-10-02T23:59:59.999Z,84880.050000000000000000,87220.000000000000000000,84520.000000000000000000,86447.530000000000000000,15148.284610000000000000,1305282019.229985000000000000,2315898,7011.464710000000000000,604129254.759916400000000000,Binance,s3://l1l2bucket/data/raw/binance/price_history/BTCUSDT/price_history_20261002T141803_901793Z_4e396c56.json,2026-10-02T14:18:03.000Z,2026-10-02T14:18:03.901Z,2026-10-02T14:23:21.544Z,2026-10-02T14:45:48.883Z,false,1567.480000000000000000,1.846700000000000000
LINKUSDT,LINK,2026-10-02T00:00:00.000Z,2026-10-02T23:59:59.999Z,14.385000000000000000,14.652000000000000000,14.164000000000000000,14.330000000000000000,2096272.630000000000000000,30191700.410280000000000000,233705,1035042.530000000000000000,14911135.702460000000000000,Binance,s3://l1l2bucket/data/raw/binance/price_history/LINKUSDT/price_history_20261002T141805_809086Z_411b4afe.json,2026-10-02T14:18:05.000Z,2026-10-02T14:18:05.809Z,2026-10-02T14:23:21.544Z,2026-10-02T14:45:48.883Z,false,-0.055000000000000000,-0.382300000000000000


In [0]:
price_keys_df = (
    silver_price_history
    .filter(
        col("is_closed") == True
    )
    .withColumn(
        "date_key",
        date_format(
            col("open_time"),
            "yyyyMMdd"
        ).cast("integer")
    )
    .withColumn(
        "exchange_key",
        upper(
            col("exchange")
        )
    )
    .withColumn(
        "asset_key",
        upper(
            col("asset")
        )
    )
    .withColumn(
        "quote_asset_key",
        lit("USDT")
    )
)


print(
    "Closed price rows:",
    price_keys_df.count()
)


display(
    price_keys_df
    .select(
        "date_key",
        "exchange_key",
        "asset_key",
        "quote_asset_key",
        "symbol",
        "open_time"
    )
    .orderBy(
        col("open_time").desc()
    )
    .limit(5)
)

Closed price rows: 2946


date_key,exchange_key,asset_key,quote_asset_key,symbol,open_time
20261001,BINANCE,SOL,USDT,SOLUSDT,2026-10-01T00:00:00.000Z
20261001,BINANCE,SUI,USDT,SUIUSDT,2026-10-01T00:00:00.000Z
20261001,BINANCE,ETH,USDT,ETHUSDT,2026-10-01T00:00:00.000Z
20261001,BINANCE,BTC,USDT,BTCUSDT,2026-10-01T00:00:00.000Z
20261001,BINANCE,ONDO,USDT,ONDOUSDT,2026-10-01T00:00:00.000Z


In [0]:
from pyspark.sql.functions import (
    col,
    concat_ws,
    current_timestamp,
    upper
)

fact_daily_prices_df = (
    price_keys_df
    .withColumn(
        "price_key",
        concat_ws(
            "|",
            col("exchange_key"),
            upper(col("symbol")),
            col("date_key").cast("string")
        )
    )
    .select(
        "price_key",
        "date_key",
        "exchange_key",
        "asset_key",
        "quote_asset_key",
        "symbol",
        "open_time",
        "close_time",
        "open_price",
        "high_price",
        "low_price",
        "close_price",
        "volume",
        "quote_asset_volume",
        "trade_count",
        "taker_buy_base_volume",
        "taker_buy_quote_volume",
        "price_change",
        "price_change_pct",
        "_source_file",
        "bronze_ingested_at"
    )
    .dropDuplicates([
        "price_key"
    ])
    .withColumn(
        "gold_processed_at",
        current_timestamp()
    )
)

print(
    "Gold daily price rows:",
    fact_daily_prices_df.count()
)

display(
    fact_daily_prices_df
    .orderBy(
        col("open_time").desc()
    )
    .limit(5)
)

Gold daily price rows: 2946


price_key,date_key,exchange_key,asset_key,quote_asset_key,symbol,open_time,close_time,open_price,high_price,low_price,close_price,volume,quote_asset_volume,trade_count,taker_buy_base_volume,taker_buy_quote_volume,price_change,price_change_pct,_source_file,bronze_ingested_at,gold_processed_at
BINANCE|SOLUSDT|20261001,20261001,BINANCE,SOL,USDT,SOLUSDT,2026-10-01T00:00:00.000Z,2026-10-01T23:59:59.999Z,118.120000000000000000,119.630000000000000000,116.730000000000000000,118.410000000000000000,2144800.225000000000000000,253033335.635520000000000000,689873,1028844.850000000000000000,121438428.637820000000000000,0.290000000000000000,0.245500000000000000,s3://l1l2bucket/data/raw/binance/price_history/SOLUSDT/price_history_20261002T141804_758297Z_2df7d944.json,2026-10-02T14:23:21.544Z,2026-10-02T16:34:29.397Z
BINANCE|SUIUSDT|20261001,20261001,BINANCE,SUI,USDT,SUIUSDT,2026-10-01T00:00:00.000Z,2026-10-01T23:59:59.999Z,1.164300000000000000,1.198600000000000000,1.127100000000000000,1.177300000000000000,74933608.900000000000000000,87280951.672700000000000000,694214,37146889.500000000000000000,43304417.222490000000000000,0.013000000000000000,1.116600000000000000,s3://l1l2bucket/data/raw/binance/price_history/SUIUSDT/price_history_20261002T141805_284183Z_b09085f0.json,2026-10-02T14:23:21.544Z,2026-10-02T16:34:29.397Z
BINANCE|ETHUSDT|20261001,20261001,BINANCE,ETH,USDT,ETHUSDT,2026-10-01T00:00:00.000Z,2026-10-01T23:59:59.999Z,2686.010000000000000000,2722.000000000000000000,2673.130000000000000000,2706.420000000000000000,263392.072900000000000000,709347817.568287000000000000,3118217,130105.991000000000000000,350428804.044927000000000000,20.410000000000000000,0.759900000000000000,s3://l1l2bucket/data/raw/binance/price_history/ETHUSDT/price_history_20261002T141804_360765Z_207c580a.json,2026-10-02T14:23:21.544Z,2026-10-02T16:34:29.397Z
BINANCE|BTCUSDT|20261001,20261001,BINANCE,BTC,USDT,BTCUSDT,2026-10-01T00:00:00.000Z,2026-10-01T23:59:59.999Z,83623.590000000000000000,85273.650000000000000000,83186.000000000000000000,84880.050000000000000000,16766.056750000000000000,1409485416.762407000000000000,3018436,8661.889790000000000000,728233115.769094700000000000,1256.460000000000000000,1.502500000000000000,s3://l1l2bucket/data/raw/binance/price_history/BTCUSDT/price_history_20261002T141803_901793Z_4e396c56.json,2026-10-02T14:23:21.544Z,2026-10-02T16:34:29.397Z
BINANCE|ONDOUSDT|20261001,20261001,BINANCE,ONDO,USDT,ONDOUSDT,2026-10-01T00:00:00.000Z,2026-10-01T23:59:59.999Z,0.499800000000000000,0.513300000000000000,0.484200000000000000,0.497300000000000000,54094942.000000000000000000,27012004.857430000000000000,116107,26953615.400000000000000000,13470047.159980000000000000,-0.002500000000000000,-0.500200000000000000,s3://l1l2bucket/data/raw/binance/price_history/ONDOUSDT/price_history_20261002T141806_205850Z_0b2ff977.json,2026-10-02T14:23:21.544Z,2026-10-02T16:34:29.397Z


In [0]:
total_rows = fact_daily_prices_df.count()
distinct = (
    fact_daily_prices_df
    .select(
        "price_key"
    )
    .distinct()
    .count()
)

duplicate = total_rows - distinct

print(
    "Total rows: ", total_rows,
    "Distinct rows: ", distinct,
    "Duplicates: ", duplicate
)

Total rows:  2946 Distinct rows:  2946 Duplicates:  0


In [0]:
(
    fact_daily_prices_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema",True)
    .saveAsTable(
        f"{catalog}.{gold_schema}.fact_daily_prices"
    )
)

####Fact daily portfolio

In [0]:
daily_balances_df = (
    spark.table(
        f"{catalog}.{gold_schema}.fact_daily_balances"
    )
)

daily_prices_df = (
    spark.table(
        f"{catalog}.{gold_schema}.fact_daily_prices"
    )
)

print("Daily balances rows: ", daily_balances_df.count())
print("Daily prices rows: ", daily_prices_df.count())

daily_balances_df.printSchema()
daily_prices_df.printSchema()

Daily balances rows:  9348
Daily prices rows:  2946
root
 |-- balance_key: string (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- balance_date: date (nullable = true)
 |-- exchange_key: string (nullable = true)
 |-- asset_key: string (nullable = true)
 |-- balance_amount: decimal(38,17) (nullable = true)
 |-- daily_change: decimal(38,18) (nullable = true)
 |-- gold_processed_at: timestamp (nullable = true)

root
 |-- price_key: string (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- exchange_key: string (nullable = true)
 |-- asset_key: string (nullable = true)
 |-- quote_asset_key: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- open_time: timestamp (nullable = true)
 |-- close_time: timestamp (nullable = true)
 |-- open_price: decimal(38,18) (nullable = true)
 |-- high_price: decimal(38,18) (nullable = true)
 |-- low_price: decimal(38,18) (nullable = true)
 |-- close_price: decimal(38,18) (nullable = true)
 |-- volume: decimal(38,18)

In [0]:
from pyspark.sql.window import Window


# Use Binance USDT prices as the reference market prices

reference_prices_df = (
    daily_prices_df
    .filter(
        (col("exchange_key") == "BINANCE")
        & (col("quote_asset_key") == "USDT")
    )
    .select(
        "date_key",
        "asset_key",
        "close_price"
    )
    .dropDuplicates([
        "date_key",
        "asset_key"
    ])
)


# Join prices by date and asset, not by the wallet's exchange

portfolio_base_df = (
    daily_balances_df
    .join(
        reference_prices_df,
        on=[
            "date_key",
            "asset_key"
        ],
        how="left"
    )
)


# Carry the latest available price forward

latest_price_window = (
    Window
    .partitionBy(
        "exchange_key",
        "asset_key"
    )
    .orderBy("date_key")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)


portfolio_base_df = (
    portfolio_base_df
    .withColumn(
        "close_price",
        last(
            "close_price",
            ignorenulls=True
        ).over(latest_price_window)
    )
)


print(
    "Portfolio rows:",
    portfolio_base_df.count()
)

print(
    "Rows without reference prices:",
    portfolio_base_df
    .filter(col("close_price").isNull())
    .count()
)

display(
    portfolio_base_df
    .filter(col("close_price").isNull())
    .select(
        "exchange_key",
        "asset_key"
    )
    .distinct()
    .orderBy(
        "exchange_key",
        "asset_key"
    )
)

Portfolio rows: 9348
Rows without reference prices: 2460


exchange_key,asset_key
BINANCE,USDT
COINDCX,INR
GIOTTUS,INR
MUDREX,INR
MUDREX,USDT


In [0]:
total_rows = portfolio_base_df.count()

missing_price_rows = (
    portfolio_base_df
    .filter(
        col("close_price").isNull()
    )
    .count()
)

print("Portfolio rows:", total_rows)
print("Rows with missing prices:", missing_price_rows)

display(
    portfolio_base_df
    .filter(
        col("close_price").isNull()
    )
    .select("asset_key")
    .distinct()
)

Portfolio rows: 9348
Rows with missing prices: 2460


asset_key
USDT
INR


In [0]:
portfolio_valued_df = (
    portfolio_base_df
    .withColumn(
        "valuation_price",
        when(
            col("asset_key") == "USDT",
            lit(1).cast("decimal(38,18)")
        ).otherwise(
            col("close_price")
        )
    )
    .withColumn(
        "asset_value_usdt",
        (
            col("balance_amount") *
            col("valuation_price")
        ).cast("decimal(38,18)")
    )
)

total_rows = portfolio_valued_df.count()

missing_prices_rows = (
    portfolio_valued_df
    .filter(
        col("valuation_price").isNull()
    )
    .count()
)

negative_prices_rows = (
    portfolio_valued_df
    .filter(
        col("valuation_price") <0
    )
    .count()
)

print("Portfolio rows:", total_rows)
print("Rows with missing prices:", missing_prices_rows)
print("Rows with negative prices:", negative_prices_rows)

Portfolio rows: 9348
Rows with missing prices: 1476
Rows with negative prices: 0


In [0]:
fact_daily_portfolio = (
    portfolio_valued_df
    .withColumn(
        "portfolio_key",
        concat_ws(
            "|",
            "exchange_key",
            "asset_key",
            "date_key"
        )
    )
    .select(
        "portfolio_key",
        "date_key",
        "balance_date",
        "exchange_key",
        "asset_key",
        "balance_amount",
        "daily_change",
        "valuation_price",
        "asset_value_usdt"
    )
    .withColumn(
        "gold_processed_at",
        current_timestamp()
    )
    .dropDuplicates(
        ["portfolio_key"]
    )
)

total_rows = fact_daily_portfolio.count()

distinct_rows = (
    fact_daily_portfolio
    .select("portfolio_key")
    .distinct()
    .count()
)

print("Daily portfolio rows:", total_rows)
print("Distinct portfolio keys:", distinct_rows)
print("Duplicates rows:", total_rows - distinct_rows)

display(
    fact_daily_portfolio
    .orderBy(
        col("balance_date").desc(),
        col("asset_value_usdt").desc()
    )
    .limit(10)
)

Daily portfolio rows: 9348
Distinct portfolio keys: 9348
Duplicates rows: 0


portfolio_key,date_key,balance_date,exchange_key,asset_key,balance_amount,daily_change,valuation_price,asset_value_usdt,gold_processed_at
MUDREX|ETH|20261002,20261002,2026-10-02,MUDREX,ETH,0.02211892543554100,0E-18,2706.420000000000000000,59.863102000000000000,2026-10-02T16:39:36.646Z
MUDREX|SOL|20261002,20261002,2026-10-02,MUDREX,SOL,0.44696715360989700,0E-18,118.410000000000000000,52.925381000000000000,2026-10-02T16:39:36.646Z
MUDREX|BTC|20261002,20261002,2026-10-02,MUDREX,BTC,0.00051788761609900,0E-18,84880.050000000000000000,43.958327000000000000,2026-10-02T16:39:36.646Z
BINANCE|BTC|20261002,20261002,2026-10-02,BINANCE,BTC,0.00028971000000000,0E-18,84880.050000000000000000,24.590599000000000000,2026-10-02T16:39:36.646Z
MUDREX|ONDO|20261002,20261002,2026-10-02,MUDREX,ONDO,38.45289965422427500,0E-18,0.497300000000000000,19.122627000000000000,2026-10-02T16:39:36.646Z
BINANCE|ETH|20261002,20261002,2026-10-02,BINANCE,ETH,0.00699300000000000,0E-18,2706.420000000000000000,18.925995000000000000,2026-10-02T16:39:36.646Z
MUDREX|SUI|20261002,20261002,2026-10-02,MUDREX,SUI,13.93681272901868400,0E-18,1.177300000000000000,16.407810000000000000,2026-10-02T16:39:36.646Z
MUDREX|LINK|20261002,20261002,2026-10-02,MUDREX,LINK,1.11054870235683200,0E-18,14.386000000000000000,15.976354000000000000,2026-10-02T16:39:36.646Z
COINDCX|SOL|20261002,20261002,2026-10-02,COINDCX,SOL,0.12194882590060000,0E-18,118.410000000000000000,14.439960000000000000,2026-10-02T16:39:36.646Z
BINANCE|SOL|20261002,20261002,2026-10-02,BINANCE,SOL,0.11488500000000000,0E-18,118.410000000000000000,13.603533000000000000,2026-10-02T16:39:36.646Z


In [0]:
(
    fact_daily_portfolio
    .write
    .mode("overwrite")
    .format("delta")
    .option("overwriteSchema", True)
    .saveAsTable(f"{catalog}.{gold_schema}.fact_daily_portfolio")
)

In [0]:
latest_balance_date = (
    saved_portfolio_df
    .agg(
        max("balance_date").alias("latest_date")
    )
    .first()["latest_date"]
)

print("Latest balance date:", latest_balance_date)

display(
    saved_portfolio_df
    .filter(
        (col("exchange_key") == "MUDREX")
        & (col("balance_date") == latest_balance_date)
    )
    .select(
        "asset_key",
        "balance_amount",
        "valuation_price",
        "asset_value_usdt"
    )
    .orderBy("asset_key")
)

Latest balance date: 2026-10-02


asset_key,balance_amount,valuation_price,asset_value_usdt
BTC,0.00051788761609900,84880.050000000000000000,43.958327000000000000
ETH,0.02211892543554100,2706.420000000000000000,59.863102000000000000
INR,0E-17,null,null
LINK,1.11054870235683200,14.386000000000000000,15.976354000000000000
ONDO,38.45289965422427500,0.497300000000000000,19.122627000000000000
SOL,0.44696715360989700,118.410000000000000000,52.925381000000000000
SUI,13.93681272901868400,1.177300000000000000,16.407810000000000000
USDT,0.75090230521755600,1.000000000000000000,0.750902000000000000


In [0]:
display(
    spark.table(
        "crypto_lakehouse.silver.mudrex_deposits"
    )
    .filter(
        col("deposit_type") == "CRYPTO"
    )
    .select(
        "deposit_id",
        "asset",
        "amount",
        "transaction_hash",
        "deposit_time"
    )
)

deposit_id,asset,amount,transaction_hash,deposit_time
019ce771-803c-7753-8c77-dae1d3bc059d,SOL,0.006414920000000000,2qLcZAvFqTytyA9cYBBreQRM9AzJoPjqrkSKY88rJhb2UYKHmSzM4LttjjNz5PZUo1AN5tSekaTf1wFQ9U9u8e4M,2026-03-13T13:45:01.000Z
019ce76c-ee3f-7118-9855-f17505cfe521,SOL,0.099999900000000000,2eMTSsX64vspd93GeSffoL84BBo8pvfDwRNpD51446sHFZH3SEpD4rg3wJJa7euTh7qwUkRY33BJgbnwyujWBYv6,2026-03-13T13:40:02.000Z
